# Amazon ML Challenge 2026 — Business Entity Resolution · FINAL (v3)

**Two-stage learned matcher + GPU ensemble, tuned for the official metric** (F0.5 per Source-1 entity,
macro-averaged, with an empty prediction for a singleton scoring 1.0).

**Run:** attach the dataset, pick the **GPU T4×2** accelerator (CPU also works), set `TEAM_NAME` in the config cell,
then **Run All**. The notebook writes:
- `/kaggle/working/output/matching_results.tsv` → upload this to the portal
- `/kaggle/working/output/candidate_pairs.tsv`
- `/kaggle/working/<TEAM_NAME>_submission.zip` → the final package (outputs, code, README, requirements,
  methodology)

### Strategy
| Phase | Where | What |
|---|---|---|
| A · train | CPU, fork pools | Hardcoded noise normalisation (Indic scripts, legal forms, OCR digits, junk, DBA, NULLs, abbreviations, FR/US/IN states) → IDF-weighted multi-key blocking, both directions, adaptive K, per-record posting budget → 75 pair/rule/context features → **stage-1 LightGBM** (GroupKFold OOF) → **learned pruning** (smallest candidate set keeping ≥ 99.8% of reachable true links) |
| B · test | CPU, fork pools | Same features; stage-1 scores each batch and prunes it immediately → **`candidate_pairs.tsv` averages ~4 candidates per S1** (a smaller set ranks higher in the final evaluation) |
| C · ensemble | **GPU**, no more forks | LightGBM + **XGBoost (CUDA)** + **CatBoost (GPU)** + **charnn**, a deep character-level cross-attention matcher (ESIM-style soft alignment over name and address bytes, fused with the tabular features, mixed precision). All are trained on the pruned pairs with the same OOF folds. The blend (single, logit-mean of all / top-3, or best GBDT + charnn) and the decision rule (threshold, or per-S1 **expected-F0.5 set**, optionally **per country**) are chosen on OOF macro F0.5. Isotonic calibration; one owner per target |

**Noise separation:** true pairs with no name, address or number evidence ("unlearnable") and non-links
indistinguishable from a link ("twins") are removed from **training only**. Validation always scores every held-out S1,
and true links lost to blocking or pruning count as misses, so the OOF number is honest.

**Deadlock and RAM safety:**
- All multiprocessing happens before CUDA is first used, and the whole pipeline runs in a fresh subprocess.
- Workers read fork-shared arrays instead of pickled copies. Each pool task has a timeout with a serial fallback.
- Blocking work per record is bounded. Test features are computed and pruned in S1 batches.

**Rules compliance:** only the provided training data is used; no external lookups. The models (LightGBM,
XGBoost, CatBoost) are MIT/Apache-2.0 and far below 8B parameters.

## 0 · Configuration

In [ ]:
TEAM_NAME = "team"          # <- your team name (used for the zip file name)
TRAIN_S1_SAMPLE = 100_000   # S1 records used for training (plenty for the models; keeps RAM and time safe)
FOLDS = 3
GPU = "auto"                # auto | on | off
RUN_SMOKE_TEST = True       # ~1 min synthetic end-to-end check before the real run
DATA_DIR = None             # None = auto-detect under /kaggle/input

## 1 · Dependencies

In [ ]:
import importlib, os, subprocess, sys
def ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg); print(f"ok        {pkg}")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True)
        try:
            importlib.import_module(mod or pkg); print(f"installed {pkg}")
        except ImportError:
            print(f"MISSING   {pkg}" + (" (built-in transliterator fallback)" if pkg == "anyascii" else ""))
for p in ["pandas", "numpy", "scikit-learn:sklearn", "lightgbm", "xgboost", "catboost", "rapidfuzz", "anyascii", "torch"]:
    n, _, m = p.partition(":"); ensure(n, m or None)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(WORK)
os.makedirs("entity_matching", exist_ok=True)
print("working dir:", WORK)
print(subprocess.run(["bash", "-c", "nvidia-smi -L 2>/dev/null || echo 'no GPU (CPU mode)'; nproc; free -g | head -2"],
                     capture_output=True, text=True).stdout)

## 2 · Pipeline source (written to `entity_matching/`)

In [ ]:
%%writefile entity_matching/__init__.py
"""Business entity resolution pipeline (Amazon ML Challenge 2026)."""

In [ ]:
%%writefile entity_matching/patterns.py
"""
Hard-coded normalisation patterns for the Amazon ML Challenge 2026
business entity-resolution task (S1 -> S2/S3 matching).

Every dictionary here comes from a noise pattern observed in the EDA
notebook (dataesplore.ipynb, cells 12-20) or from the countries present
in the test set (US, India, and the unseen France).

All keys are lower-case, ASCII-folded, punctuation-free unless noted.
"""

# ---------------------------------------------------------------------------
# Placeholders that mean "no value" (cell 14: null / <null> / none / nan)
# ---------------------------------------------------------------------------
NULL_TOKENS = {
    "null", "<null>", "none", "nan", "n/a", "na", "nil", "unknown",
    "not available", "not applicable", "-", "--", "0",
}

# ---------------------------------------------------------------------------
# Legal / entity forms -> canonical form.
# Multi-word variants are matched on the space-normalised string first.
# ---------------------------------------------------------------------------
LEGAL_MULTIWORD = [
    # order matters: longest first
    ("limited liability partnership", "llp"),
    ("limited liability company", "llc"),
    ("professional limited liability company", "pllc"),
    ("professional corporation", "pc"),
    ("limited partnership", "lp"),
    ("private limited company", "pvt ltd"),
    ("private limited", "pvt ltd"),
    ("public limited company", "public ltd"),
    ("public limited", "public ltd"),
    ("one person company", "opc"),
    ("societe anonyme", "sa"),
    ("societe par actions simplifiee unipersonnelle", "sasu"),
    ("societe par actions simplifiee", "sas"),
    ("societe a responsabilite limitee", "sarl"),
    ("entreprise unipersonnelle a responsabilite limitee", "eurl"),
    ("societe civile immobiliere", "sci"),
    ("et compagnie", "cie"),
    ("et cie", "cie"),
    ("and company", "co"),
    ("and co", "co"),
    ("pvt ltd", "pvt ltd"),
]

LEGAL_TOKENS = {
    # US
    "llc": "llc", "l.l.c": "llc", "llc.": "llc", "lllc": "llc",
    "inc": "inc", "incorporated": "inc", "incorp": "inc", "inco": "inc",
    "corp": "corp", "corporation": "corp", "corpn": "corp",
    "co": "co", "company": "co", "cos": "co",
    "ltd": "ltd", "limited": "ltd", "ltda": "ltd", "lmtd": "ltd", "limted": "ltd",
    "llp": "llp", "lp": "lp", "pllc": "pllc", "pc": "pc", "pa": "pa",
    "plc": "plc", "lc": "lc", "ltd.": "ltd",
    # India
    "pvt": "pvt", "private": "pvt", "pvtltd": "pvt ltd", "prv": "pvt",
    "public": "public", "opc": "opc",
    # France (unseen in train -> hard-coded)
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "sa": "sa", "eurl": "eurl",
    "sci": "sci", "snc": "snc", "scp": "scp", "selarl": "selarl",
    "scop": "scop", "gie": "gie", "cie": "cie", "compagnie": "cie",
    "ets": "ets", "etablissements": "ets", "etablissement": "ets",
    "societe": "societe", "ste": "societe",
    # Generic international
    "gmbh": "gmbh", "ag": "ag", "bv": "bv", "nv": "nv", "srl": "srl",
    "spa": "spa", "oy": "oy", "ab": "ab", "pty": "pty",
}

# Native-script legal words (cells 13/15: Indic-script names keep the legal
# form transliterated phonetically).  Replaced BEFORE transliteration.
INDIC_LEGAL = {
    # Devanagari (Hindi / Marathi)
    "प्राइवेट": " private ", "प्रायवेट": " private ", "प्राईवेट": " private ",
    "लिमिटेड": " limited ", "लिमीटेड": " limited ",
    "प्रा.": " pvt ", "प्रा": " pvt ", "लि.": " ltd ", "लि": " ltd ",
    "एलएलपी": " llp ", "कंपनी": " company ", "कम्पनी": " company ",
    "इंक": " inc ", "कॉर्प": " corp ",
    # Bengali
    "প্রাইভেট": " private ", "লিমিটেড": " limited ", "প্রা.": " pvt ",
    "লি.": " ltd ", "এলএলপি": " llp ",
    # Tamil
    "பிரைவேட்": " private ", "லிமிடெட்": " limited ", "எல்எல்பி": " llp ",
    # Telugu
    "ప్రైవేట్": " private ", "లిమిటెడ్": " limited ", "ఎల్ఎల్పి": " llp ",
    # Kannada
    "ಪ್ರೈವೇಟ್": " private ", "ಲಿಮಿಟೆಡ್": " limited ", "ಎಲ್ಎಲ್ಪಿ": " llp ",
    # Gujarati
    "પ્રાઇવેટ": " private ", "લિમિટેડ": " limited ", "પ્રા.": " pvt ",
    "લિ.": " ltd ", "એલએલપી": " llp ",
    # Malayalam
    "പ്രൈവറ്റ്": " private ", "ലിമിറ്റഡ്": " limited ", "എൽഎൽപി": " llp ",
    # Punjabi (Gurmukhi) / Odia – rarer but cheap to include
    "ਪ੍ਰਾਈਵੇਟ": " private ", "ਲਿਮਿਟੇਡ": " limited ",
    "ପ୍ରାଇଭେଟ": " private ", "ଲିମିଟେଡ": " limited ",
}

# Phonetic skeletons of legal words: catches transliterations such as
# "praivet", "piraivet", "limitet", "limitedd" (see normalize.skeleton()).
LEGAL_SKELETON_WORDS = [
    "private", "limited", "pvt", "ltd", "llp", "company", "incorporated",
    "corporation", "praivet", "piraivet", "limitet", "elelpi",
]

# DBA / alias markers (cells 15-16): "X D.B.A. Y", "Korzephdova formerly: Y",
# "Viovera | www.viovera.com".
ALIAS_MARKERS = [
    r"\bd\s*/\s*b\s*/\s*a\b", r"\bd\.?\s?b\.?\s?a\.?\b", r"\bdba\b",
    r"\bt\s*/\s*a\b", r"\btrading\s+as\b", r"\bdoing\s+business\s+as\b",
    r"\bformerly\s+known\s+as\b", r"\bformerly\b", r"\bf\s*/\s*k\s*/\s*a\b",
    r"\bfka\b", r"\ba\s*/\s*k\s*/\s*a\b", r"\baka\b", r"\balso\s+known\s+as\b",
    r"\bsucc(?:essor)?\s+to\b", r"\bnee\b", r"\|",
]

# Junk that the noisy sources prepend/append (cells 13-15).
JUNK_REGEXES = [
    r"\(\s*id\s*[:#]?\s*\d+\s*\)",      # (ID: 49166)
    r"\bid\s*[:#]\s*\d+\b",              # ID: 49166
    r"#\s*\d{3,}\b",                     # #98587
    r"^\W+",                             # ***, --, >>, # at start
    r"\W+$",                             # trailing symbols
    r"\bm\s*/\s*s\b\.?",                  # M/s  (Indian "Messrs"), anywhere
    r"^\s*messrs\.?\s+",
    r"\bwww\.",                          # www.
    r"https?://",
]

# Words that carry no identity in a business name.
NAME_STOPWORDS = {
    "the", "of", "and", "a", "an", "for", "at", "in", "on", "by",
    # French
    "de", "la", "le", "les", "du", "des", "et", "l", "d", "au", "aux",
    # Indian honorifics that noisy sources add/drop ("Sri", "Shri", "M/s")
    "m/s", "ms", "messrs",
}
HONORIFICS = {"sri", "shri", "shree", "smt", "dr", "mr", "mrs", "st"}

# Top-level domains stripped from names written as URLs ("aahanarealty.com").
TLDS = {
    "com", "net", "org", "in", "co", "io", "biz", "info", "us", "fr",
    "co.in", "org.in", "net.in", "edu", "gov", "c0m", "corn",
}

# ---------------------------------------------------------------------------
# OCR / "leet" confusions inside alphabetic tokens (cell 15/16: 5outh,
# WASHINGT0N, lnc, FAMlLY, silvergrill.c0m, Dermato1ogy, MEMORlAL).
# Applied symmetrically to BOTH sides to build an OCR-folded comparison key.
# ---------------------------------------------------------------------------
OCR_DIGIT_TO_CHAR = {
    "0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "6": "g", "7": "t",
    "8": "b", "9": "g", "@": "a", "$": "s", "|": "l", "!": "l",
}
# After digit fixing, collapse visually confusable letters.
OCR_CHAR_FOLD = [("rn", "m"), ("vv", "w"), ("cl", "d"), ("l", "i"), ("j", "i")]

# ---------------------------------------------------------------------------
# Address: US states (name <-> USPS code)
# ---------------------------------------------------------------------------
US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar",
    "california": "ca", "colorado": "co", "connecticut": "ct", "delaware": "de",
    "district of columbia": "dc", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in",
    "iowa": "ia", "kansas": "ks", "kentucky": "ky", "louisiana": "la",
    "maine": "me", "maryland": "md", "massachusetts": "ma", "michigan": "mi",
    "minnesota": "mn", "mississippi": "ms", "missouri": "mo", "montana": "mt",
    "nebraska": "ne", "nevada": "nv", "new hampshire": "nh", "new jersey": "nj",
    "new mexico": "nm", "new york": "ny", "north carolina": "nc",
    "north dakota": "nd", "ohio": "oh", "oklahoma": "ok", "oregon": "or",
    "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut",
    "vermont": "vt", "virginia": "va", "washington": "wa",
    "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy",
    "puerto rico": "pr", "guam": "gu", "virgin islands": "vi",
}

# ---------------------------------------------------------------------------
# Address: Indian states / UTs (name, common code, native-script spelling)
# ---------------------------------------------------------------------------
INDIA_STATES = {
    "andhra pradesh": "ap", "arunachal pradesh": "ar", "assam": "as",
    "bihar": "br", "chhattisgarh": "cg", "chattisgarh": "cg", "goa": "ga",
    "gujarat": "gj", "haryana": "hr", "himachal pradesh": "hp",
    "jharkhand": "jh", "karnataka": "ka", "kerala": "kl", "keralam": "kl",
    "madhya pradesh": "mp", "maharashtra": "mh", "manipur": "mn",
    "meghalaya": "ml", "mizoram": "mz", "nagaland": "nl", "odisha": "od",
    "orissa": "od", "punjab": "pb", "rajasthan": "rj", "sikkim": "sk",
    "tamil nadu": "tn", "tamilnadu": "tn", "telangana": "tg", "tripura": "tr",
    "uttar pradesh": "up", "uttarakhand": "uk", "uttaranchal": "uk",
    "west bengal": "wb", "delhi": "dl", "new delhi": "dl", "nct of delhi": "dl",
    "jammu and kashmir": "jk", "jammu kashmir": "jk", "ladakh": "la",
    "chandigarh": "ch", "puducherry": "py", "pondicherry": "py",
    "andaman and nicobar islands": "an", "lakshadweep": "ld",
    "dadra and nagar haveli": "dn", "daman and diu": "dd",
}
INDIA_STATE_CODE_ALIASES = {"ts": "tg", "or": "od", "ua": "uk", "ct": "cg"}

# Native-script state names seen in S2/S3 addresses (cell 13).
INDIA_STATES_NATIVE = {
    "महाराष्ट्र": "mh", "दिल्ली": "dl", "राजस्थान": "rj", "मध्य प्रदेश": "mp",
    "उत्तर प्रदेश": "up", "बिहार": "br", "हरियाणा": "hr", "पंजाब": "pb",
    "उत्तराखंड": "uk", "झारखंड": "jh", "छत्तीसगढ़": "cg", "हिमाचल प्रदेश": "hp",
    "गोवा": "ga", "ಕರ್ನಾಟಕ": "ka", "தமிழ்நாடு": "tn", "తెలంగాణ": "tg",
    "ఆంధ్ర ప్రదేశ్": "ap", "ఆంధ్రప్రదేశ్": "ap", "পশ্চিমবঙ্গ": "wb",
    "ગુજરાત": "gj", "കേരളം": "kl", "ଓଡ଼ିଶା": "od", "ਪੰਜਾਬ": "pb",
    "অসম": "as", "आसाम": "as",
}

# ---------------------------------------------------------------------------
# France regions / departments present in test (cell 12, descriptive only)
# ---------------------------------------------------------------------------
FRANCE_REGIONS = {
    "hauts de france": "hdf", "nouvelle aquitaine": "naq",
    "pays de la loire": "pdl", "ile de france": "idf", "bretagne": "bre",
    "normandie": "nor", "grand est": "ges", "occitanie": "occ",
    "auvergne rhone alpes": "ara", "provence alpes cote d azur": "pac",
    "bourgogne franche comte": "bfc", "centre val de loire": "cvl",
    "corse": "cor", "nord": "d59", "pas de calais": "d62", "gironde": "d33",
    "loire atlantique": "d44",
}

# ---------------------------------------------------------------------------
# City aliases (old/new names, common misspellings)
# ---------------------------------------------------------------------------
CITY_ALIASES = {
    "bengaluru": "bangalore", "bangaluru": "bangalore", "bombay": "mumbai",
    "madras": "chennai", "calcutta": "kolkata", "gurugram": "gurgaon",
    "thiruvananthapuram": "trivandrum", "poona": "pune", "baroda": "vadodara",
    "prayagraj": "allahabad", "vizag": "visakhapatnam",
    "vishakhapatnam": "visakhapatnam", "vishakapatnam": "visakhapatnam",
    "mysuru": "mysore", "mangaluru": "mangalore", "cochin": "kochi",
    "belagavi": "belgaum", "kalaburagi": "gulbarga", "cawnpore": "kanpur",
    "benares": "varanasi", "banaras": "varanasi", "simla": "shimla",
    "ahmedbad": "ahmedabad", "amdavad": "ahmedabad", "secunderabad": "hyderabad",
    "hubballi": "hubli", "tiruchirappalli": "trichy", "tiruchirapalli": "trichy",
    "puducherry": "pondicherry", "nw delhi": "new delhi", "navi mumbai": "navi mumbai",
    "gautam budh nagar": "gautam buddha nagar", "goutam budd nagar": "gautam buddha nagar",
    "noida": "gautam buddha nagar",
}

# ---------------------------------------------------------------------------
# Street / thoroughfare types (USPS Pub 28 + Indian + French) -> canonical
# ---------------------------------------------------------------------------
STREET_TYPES = {
    # US
    "street": "st", "str": "st", "st": "st", "saint": "st",  # "25 PALMER SAINT"
    "road": "rd", "rd": "rd", "avenue": "ave", "ave": "ave", "av": "ave",
    "aven": "ave", "avn": "ave", "drive": "dr", "dr": "dr", "drv": "dr",
    "lane": "ln", "ln": "ln", "court": "ct", "ct": "ct", "crt": "ct",
    "boulevard": "blvd", "blvd": "blvd", "boul": "blvd", "bd": "blvd",
    "place": "pl", "pl": "pl", "terrace": "ter", "ter": "ter", "terr": "ter",
    "trail": "trl", "trl": "trl", "circle": "cir", "cir": "cir", "circ": "cir",
    "parkway": "pkwy", "pkwy": "pkwy", "pky": "pkwy", "highway": "hwy",
    "hwy": "hwy", "square": "sq", "sq": "sq", "way": "way", "wy": "way",
    "expressway": "expy", "expy": "expy", "freeway": "fwy", "fwy": "fwy",
    "turnpike": "tpke", "tpke": "tpke", "crossing": "xing", "xing": "xing",
    "point": "pt", "pt": "pt", "pointe": "pt", "heights": "hts", "hts": "hts",
    "mount": "mt", "mt": "mt", "mountain": "mtn", "mtn": "mtn",
    "center": "ctr", "centre": "ctr", "ctr": "ctr", "plaza": "plz", "plz": "plz",
    "alley": "aly", "aly": "aly", "loop": "loop", "run": "run", "row": "row",
    "pike": "pike", "path": "path", "walk": "walk", "cove": "cv", "cv": "cv",
    "creek": "crk", "crk": "crk", "ridge": "rdg", "rdg": "rdg",
    "hollow": "holw", "holw": "holw", "valley": "vly", "vly": "vly",
    "junction": "jct", "jct": "jct", "estates": "est", "est": "est",
    "station": "sta", "sta": "sta", "township": "twp", "twp": "twp",
    "townhsip": "twp",
    # India
    "marg": "marg", "mg": "marg", "nagar": "nagar", "ngr": "nagar",
    "colony": "colony", "col": "colony", "sector": "sector", "sec": "sector",
    "phase": "phase", "ph": "phase", "block": "block", "blk": "block",
    "layout": "layout", "extension": "extn", "extn": "extn", "ext": "extn",
    "chowk": "chowk", "gali": "gali", "mohalla": "mohalla", "bazar": "bazaar",
    "bazaar": "bazaar", "cross": "cross", "main": "main", "salai": "salai",
    "industrial": "indl", "indl": "indl", "ind": "indl", "estate": "est",
    "complex": "cplx", "cplx": "cplx", "building": "bldg", "bldg": "bldg",
    "bldng": "bldg", "tower": "twr", "twr": "twr", "apartment": "apt",
    "apartments": "apt", "apts": "apt", "apt": "apt", "society": "soc",
    "soc": "soc", "chs": "soc", "near": "nr", "nr": "nr", "opposite": "opp",
    "opp": "opp", "behind": "bhd", "bh": "bhd", "post": "po", "po": "po",
    "village": "vill", "vill": "vill", "vil": "vill", "taluka": "tal",
    "tal": "tal", "tehsil": "tal", "district": "dist", "dist": "dist",
    "distt": "dist", "railway": "rly", "rly": "rly",
    # France
    "rue": "rue", "r": "rue", "allee": "all", "all": "all", "impasse": "imp",
    "imp": "imp", "chemin": "che", "che": "che", "ch": "che", "route": "rte",
    "rte": "rte", "quai": "quai", "cours": "crs", "crs": "crs", "residence": "res",
    "res": "res", "lieu dit": "ld", "lieudit": "ld", "ld": "ld", "faubourg": "fbg",
    "fbg": "fbg", "passage": "pass", "pass": "pass", "sentier": "sen",
    "hameau": "ham", "ham": "ham", "zone": "zone", "za": "za", "zi": "zi",
    "zac": "zac", "bis": "bis", "ter.": "ter", "sainte": "ste", "ste": "ste",
    "cedex": "",
}

DIRECTIONALS = {
    "north": "n", "south": "s", "east": "e", "west": "w", "northeast": "ne",
    "northwest": "nw", "southeast": "se", "southwest": "sw", "n": "n",
    "s": "s", "e": "e", "w": "w", "ne": "ne", "nw": "nw", "se": "se",
    "sw": "sw", "nord": "n", "sud": "s", "est": "e", "ouest": "w",
}

# Unit / house labels that precede a number (cell 14): the label itself is
# dropped, the number is kept.
UNIT_LABELS = {
    "unit", "apt", "apartment", "suite", "ste", "room", "rm", "floor", "fl",
    "flr", "office", "off", "shop", "plot", "flat", "house", "door", "no",
    "number", "num", "nos", "hno", "h.no", "hn", "dno", "d.no", "fno", "f.no",
    "khasra", "kh", "survey", "sr", "sy", "s.no", "sno", "ward", "wing",
    "bldg", "building", "block", "pmb", "box", "po box", "pobox", "#",
    "premises", "cabin", "gala", "stall", "site", "khata", "cts", "rs",
    "gat", "gut", "pl", "bat", "batiment", "etage", "appt", "porte",
}

ORDINAL_WORDS = {
    "first": "1", "second": "2", "third": "3", "fourth": "4", "fifth": "5",
    "sixth": "6", "seventh": "7", "eighth": "8", "ninth": "9", "tenth": "10",
    "eleventh": "11", "twelfth": "12", "ground": "0", "gf": "0",
    "premier": "1", "deuxieme": "2", "troisieme": "3", "rdc": "0",
}

ADDRESS_STOPWORDS = {
    "of", "the", "and", "at", "in", "near", "city", "town", "de", "la", "le",
    "du", "des", "les", "l", "d", "s/o", "c/o", "w/o", "d/o", "county",
    "region", "india", "usa", "us", "france", "united", "states",
}

In [ ]:
%%writefile entity_matching/normalize.py
"""
Name / address normalisation.

Every function is pure (str -> dict of str) so it can be mapped over the
unique values of a column with multiprocessing.
"""
from __future__ import annotations

import re
import unicodedata
from functools import lru_cache

from . import patterns as P

try:  # best transliterator for all Indic scripts + accent folding
    from anyascii import anyascii as _anyascii
except ImportError:  # pragma: no cover - fallback keeps the pipeline running
    _anyascii = None

# ---------------------------------------------------------------------------
# Unicode helpers
# ---------------------------------------------------------------------------
_ZERO_WIDTH = re.compile(r"[​-‍﻿­]")
INDIC_RE = re.compile(r"[ऀ-෿]")
_NON_ASCII = re.compile(r"[^\x00-\x7F]")
_WS = re.compile(r"\s+")

_INDIC_LEGAL_SORTED = sorted(P.INDIC_LEGAL.items(), key=lambda kv: -len(kv[0]))
_NATIVE_STATES_SORTED = sorted(P.INDIA_STATES_NATIVE.items(), key=lambda kv: -len(kv[0]))


def _fallback_translit(text: str) -> str:
    """Transliterate without anyascii: accents via NFKD, Brahmic scripts via
    Unicode character names ("DEVANAGARI LETTER RA" -> "ra")."""
    out = []
    for ch in unicodedata.normalize("NFKD", text):
        if ord(ch) < 128:
            out.append(ch)
            continue
        if unicodedata.combining(ch) and not INDIC_RE.match(ch):
            continue
        name = unicodedata.name(ch, "")
        if " LETTER " in name:
            syl = name.split(" LETTER ")[-1].lower().split()[0]
            out.append(syl if len(syl) <= 3 else syl[:3])
        elif " VOWEL SIGN " in name:
            out.append(name.split(" VOWEL SIGN ")[-1].lower().split()[0][:2])
        elif " DIGIT " in name:
            out.append(str(unicodedata.digit(ch, 0)))
        elif "VIRAMA" in name and out and out[-1].endswith("a"):
            out[-1] = out[-1][:-1]
        else:
            out.append(" ")
    return "".join(out)


def to_ascii(text: str) -> str:
    if not _NON_ASCII.search(text):
        return text
    if _anyascii is not None:
        return _anyascii(text)
    return _fallback_translit(text)


def clean_unicode(text) -> str:
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = _ZERO_WIDTH.sub("", text)
    return text.strip()


# ---------------------------------------------------------------------------
# Phonetic skeleton - makes transliterated Indic names comparable with the
# English spelling ("sliusns" ~ "solutions", "piraivet" ~ "private").
# ---------------------------------------------------------------------------
_SKEL_SUBS = [
    ("tion", "sn"), ("sion", "sn"), ("tian", "sn"), ("ph", "f"), ("sh", "s"),
    ("ch", "c"), ("th", "t"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
    ("dh", "d"), ("jh", "j"), ("ck", "k"), ("qu", "k"), ("x", "ks"),
    ("w", "v"), ("z", "s"), ("ce", "se"), ("ci", "si"), ("cy", "sy"),
    ("ge", "je"), ("gi", "ji"), ("gy", "jy"),
    ("c", "k"), ("q", "k"), ("b", "p"), ("d", "t"),
    ("g", "k"), ("m", "n"),
]
_VOWELS = set("aeiouyh")


@lru_cache(maxsize=100_000)
def skeleton(token: str) -> str:
    if not token:
        return ""
    if token.isdigit():
        return token.lstrip("0") or "0"
    t = token
    for a, b in _SKEL_SUBS:
        t = t.replace(a, b)
    head, rest = t[0], [c for c in t[1:] if c not in _VOWELS]
    out = [head]
    for c in rest:
        if c != out[-1]:
            out.append(c)
    return "".join(out)


_LEGAL_SKELETONS = {skeleton(w) for w in P.LEGAL_SKELETON_WORDS}


def ocr_fold(text: str) -> str:
    """Symmetric OCR folding for comparison keys only."""
    out = []
    for tok in text.split():
        if any(c.isalpha() for c in tok):
            tok = "".join(P.OCR_DIGIT_TO_CHAR.get(c, c) for c in tok)
        for a, b in P.OCR_CHAR_FOLD:
            tok = tok.replace(a, b)
        out.append(tok)
    return " ".join(out)


# ---------------------------------------------------------------------------
# Business names
# ---------------------------------------------------------------------------
_ALIAS_RE = re.compile("|".join(P.ALIAS_MARKERS))
_JUNK_RES = [re.compile(r) for r in P.JUNK_REGEXES]
_DOTTED_ABBR = re.compile(r"\b(?:[a-z]\.){2,}[a-z]?\.?")
_DOMAIN_RE = re.compile(
    r"^(?:www\.)?([a-z0-9][a-z0-9\-]*?)\.(" + "|".join(
        re.escape(t) for t in sorted(P.TLDS, key=len, reverse=True)) + r")\.?$")
_NAME_PUNCT = re.compile(r"[^a-z0-9\s]")
_LEGAL_MULTI_MAP = dict(P.LEGAL_MULTIWORD)
_LEGAL_MULTI_RE = re.compile(r"\b(" + "|".join(
    re.escape(a) for a in sorted(_LEGAL_MULTI_MAP, key=lambda k: -len(k))) + r")\b")
_LEGAL_TOKENS = dict(P.LEGAL_TOKENS, lnc="inc", ilc="llc", iic="llc", ltdd="ltd")


def _fix_ocr_token(tok: str) -> str:
    # digits inside a word are OCR noise: 5outh, washingt0n, dermato1ogy
    if any(c.isdigit() for c in tok) and sum(c.isalpha() for c in tok) >= 2:
        return "".join(P.OCR_DIGIT_TO_CHAR.get(c, c) for c in tok)
    return tok


def _clean_single_name(s: str, from_indic: bool):
    flags = {"domain": 0, "junk": 0}
    s = s.strip()
    for rx in _JUNK_RES:
        new = rx.sub(" ", s)
        if new != s:
            flags["junk"] = 1
            s = new
    s = s.strip()
    # whole name written as a web domain: aahanarealty.com / silvergrill.c0m
    m = _DOMAIN_RE.match(s.replace(" ", ""))
    if m and " " not in s.strip():
        s = m.group(1).replace("-", " ")
        flags["domain"] = 1
    s = _DOTTED_ABBR.sub(lambda mm: mm.group(0).replace(".", ""), s)
    s = s.replace("&", " and ").replace("+", " and ").replace("'s ", "s ")
    s = s.replace("'", "")
    s = _NAME_PUNCT.sub(" ", s)
    s = _WS.sub(" ", s).strip()
    s = _LEGAL_MULTI_RE.sub(lambda m: _LEGAL_MULTI_MAP[m.group(1)], s)

    core, legal = [], set()
    for tok in s.split():
        tok = _fix_ocr_token(tok)
        if tok in _LEGAL_TOKENS:
            for lg in _LEGAL_TOKENS[tok].split():
                legal.add(lg)
            continue
        if from_indic and len(tok) > 2 and skeleton(tok) in _LEGAL_SKELETONS:
            legal.add(tok)
            continue
        if tok in P.NAME_STOPWORDS or tok in P.HONORIFICS:
            continue
        if core and core[-1] == tok:  # repeated word: "bright bright"
            continue
        core.append(tok)
    if not core and legal:  # name was only legal words - keep something
        core = sorted(legal)
    return core, legal, flags


@lru_cache(maxsize=20_000)
def normalize_name(raw) -> dict:
    s = clean_unicode(raw)
    has_indic = bool(INDIC_RE.search(s))
    if has_indic:
        for native, rep in _INDIC_LEGAL_SORTED:
            s = s.replace(native, rep)
    s = to_ascii(s).lower()

    parts = [p for p in _ALIAS_RE.split(s) if p and p.strip()]
    if not parts:
        parts = [s]
    alts = []
    legal_all: set = set()
    flags = {"domain": 0, "junk": 0}
    for p in parts:
        core, legal, fl = _clean_single_name(p, has_indic)
        if core:
            alts.append(core)
        legal_all |= legal
        for k in flags:
            flags[k] = max(flags[k], fl[k])
    if not alts:
        alts = [[]]
    # primary = the longest alternative (usually the real business name)
    main = max(alts, key=lambda c: len(" ".join(c)))
    core_str = " ".join(main)
    skel = " ".join(skeleton(t) for t in main)
    return {
        "core": core_str,
        "full": (core_str + " " + " ".join(sorted(legal_all))).strip(),
        "ocr": ocr_fold(core_str),
        "skel": skel,
        "nospace": core_str.replace(" ", ""),
        "alts": "||".join(dict.fromkeys(" ".join(a) for a in alts if a)) if len(alts) > 1 else "",
        "legal": " ".join(sorted(legal_all)),
        "acr": "".join(t[0] for t in main) if len(main) >= 2 else "",
        "indic": int(has_indic),
        "domain": flags["domain"],
        "junk": int(flags["junk"] or len(alts) > 1),
    }


# ---------------------------------------------------------------------------
# Addresses
# ---------------------------------------------------------------------------
_NULL_RE = re.compile(r"<\s*null\s*>|\b(?:null|none|nan|n/a)\b", re.I)
_ADDR_PUNCT = re.compile(r"[^a-z0-9\s/\-#,]")
_ORD_SUFFIX = re.compile(r"^(\d+)(st|nd|rd|th|er|e|eme|re)$")
_DIGITS = re.compile(r"\d+")
_LABEL_GLUE = re.compile(r"\b(h|d|f|s|sy|kh|r|p)\s*\.\s*no(?=\b|\d)")  # h.no -> hno
_LABEL_DIGIT = re.compile(r"\b(no|hno|dno|fno|sno|plot|flat|door|shop|unit|apt|suite|ste|room|office|house)(?=\d)")
_WORD_HYPHEN = re.compile(r"(?<=[a-z])-(?=[a-z])")
_FRANCE_ONLY = {"r", "ch", "all", "che", "imp", "ld", "res"}
_NO_GLUE = re.compile(r"\bno\s*[.:]\s*")
_HASH = re.compile(r"#+\s*")

_US_CODES = set(P.US_STATES.values())
_IN_CODES = set(P.INDIA_STATES.values()) | set(P.INDIA_STATE_CODE_ALIASES)
_ALL_STATE_NAMES = sorted(
    list(P.US_STATES.items()) + list(P.INDIA_STATES.items()) + list(P.FRANCE_REGIONS.items()),
    key=lambda kv: -len(kv[0]))
# one longest-first alternation per table instead of ~110 / ~30 separate
# regex passes per address (5-10x faster on 9M addresses)
_STATE_MAP = dict(_ALL_STATE_NAMES)
_STATE_RE = re.compile(r"\b(" + "|".join(re.escape(n) for n, _ in _ALL_STATE_NAMES) + r")\b")
_CITY_MAP = dict(P.CITY_ALIASES)
_CITY_RE = re.compile(r"\b(" + "|".join(
    re.escape(a) for a in sorted(P.CITY_ALIASES, key=lambda k: -len(k))) + r")\b")
_STATE_CODES = _US_CODES | _IN_CODES | set(P.FRANCE_REGIONS.values())


def canon_number(tok: str) -> str:
    """0271a -> 271a, 4Nd -> 4, 05131 -> 5131, b-204 -> b-204."""
    m = _ORD_SUFFIX.match(tok)
    if m:
        return m.group(1).lstrip("0") or "0"
    return _DIGITS.sub(lambda mm: mm.group(0).lstrip("0") or "0", tok).strip("-/")


@lru_cache(maxsize=20_000)
def normalize_address(raw, country: str = "") -> dict:
    s = clean_unicode(raw)
    for native, code in _NATIVE_STATES_SORTED:
        if native in s:
            s = s.replace(native, f" {code} ")
    s = to_ascii(s).lower()
    s = _NULL_RE.sub(" ", s)
    s = _LABEL_GLUE.sub(lambda m: m.group(1) + "no ", s)
    s = _NO_GLUE.sub("no ", s)
    s = _LABEL_DIGIT.sub(lambda m: m.group(1) + " ", s)
    s = _WORD_HYPHEN.sub(" ", s)
    s = _HASH.sub(" # ", s)
    s = s.replace(".", " ").replace(";", ",").replace("(", " ").replace(")", " ")
    s = _ADDR_PUNCT.sub(" ", s)
    s = _CITY_RE.sub(lambda m: _CITY_MAP[m.group(1)], s)

    alpha, nums, numparts, numstreet, states = [], [], set(), [], set()
    ordered = []
    for comp in s.split(","):
        comp = _WS.sub(" ", comp).strip(" -/")
        if not comp:
            continue
        if comp in _STATE_CODES and (
                (country == "US" and comp in _US_CODES)
                or (country == "India" and (comp in _IN_CODES))
                or country not in ("US", "India")):
            states.add(P.INDIA_STATE_CODE_ALIASES.get(comp, comp) if country == "India" else comp)
            continue
        if _STATE_RE.search(comp):
            for m in _STATE_RE.finditer(comp):
                states.add(_STATE_MAP[m.group(1)])
            comp = _STATE_RE.sub(" ", comp)
        toks = comp.split()
        prev_num = None
        for i, tok in enumerate(toks):
            tok = tok.strip("-/")
            if not tok or tok == "#":
                continue
            if tok in P.ORDINAL_WORDS and i + 1 < len(toks) and toks[i + 1] in ("floor", "fl", "flr", "etage"):
                tok = P.ORDINAL_WORDS[tok]
            if any(c.isdigit() for c in tok):
                cn = canon_number(tok)
                if cn:
                    nums.append(cn)
                    numparts.update(d.lstrip("0") or "0" for d in _DIGITS.findall(tok))
                    ordered.append(cn)
                    prev_num = cn
                continue
            if tok in P.UNIT_LABELS or tok in P.ADDRESS_STOPWORDS:
                continue
            if country == "France" or tok not in _FRANCE_ONLY:
                tok = P.STREET_TYPES.get(tok, tok)
            tok = P.DIRECTIONALS.get(tok, tok)
            if not tok:
                continue
            if len(tok) == 2 and tok in _STATE_CODES and i == len(toks) - 1 and len(toks) <= 2:
                states.add(tok)
                continue
            alpha.append(tok)
            ordered.append(tok)
            if prev_num is not None:
                numstreet.append(f"{prev_num}_{tok}")
                prev_num = None

    postal = sorted(n for n in nums if n.isdigit() and len(n) in (5, 6))
    return {
        "atext": " ".join(ordered),
        "alpha": " ".join(alpha),
        "nums": " ".join(nums),
        "numparts": " ".join(sorted(numparts)),
        "numstreet": " ".join(numstreet),
        "first_num": nums[0] if nums else "",
        "state": " ".join(sorted(states)),
        "postal": " ".join(postal),
        "missing": int(not ordered),
    }

In [ ]:
%%writefile entity_matching/utils.py
"""Kaggle helpers: path auto-detection, deadlock-safe process pools, memory logs."""
from __future__ import annotations

import gc
import os
import sys
import time
from array import array
from multiprocessing import get_context
from multiprocessing.context import TimeoutError as MPTimeout
from pathlib import Path

import numpy as np

_T0 = time.time()

# Keep native thread pools single-threaded inside forked workers: forking a
# process whose OpenMP/BLAS pool is live and then using that pool in the
# child is the classic multiprocessing deadlock.  Set before numpy/lightgbm
# spin their pools up in the children.
_THREAD_VARS = ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS")

REQUIRED = ("train_source1.tsv", "train_source2.tsv", "train_source3.tsv")
SEARCH_ROOTS = ("/kaggle/input", "/kaggle/working", ".", "./dataset", "./data", "..")


def rss_gb() -> float:
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024 ** 2
    except OSError:
        pass
    return float("nan")


def log(msg: str) -> None:
    print(f"[{time.time() - _T0:7.1f}s | {rss_gb():5.1f} GB] {msg}", flush=True)


def free() -> None:
    gc.collect()


def is_kaggle() -> bool:
    return Path("/kaggle/working").is_dir() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ


def auto_data_dir(explicit: str | None = None, split: str = "train") -> Path:
    """Find the folder tree holding the challenge TSVs.

    Checks the explicit path, $EM_DATA_DIR, then searches /kaggle/input (any
    dataset/competition slug, any nesting depth) and local folders for
    `{split}_source1.tsv`.  Returns the common root that contains both the
    train and test files when possible."""
    cands = [explicit, os.environ.get("EM_DATA_DIR")] + list(SEARCH_ROOTS)
    for root in cands:
        if not root:
            continue
        root = Path(root)
        if not root.is_dir():
            continue
        hits = sorted(root.rglob(f"{split}_source1.tsv"))
        if not hits:
            continue
        # prefer a root that also contains the other split
        for h in hits:
            for parent in [h.parent, h.parent.parent]:
                names = {p.name for p in parent.rglob("*.tsv")}
                if {"train_source1.tsv", "test_source1.tsv"} <= names:
                    return parent
        return hits[0].parent.parent if hits[0].parent.name == split else hits[0].parent
    raise FileNotFoundError(
        f"Could not find {split}_source1.tsv under {', '.join(str(c) for c in cands if c)}. "
        "Attach the dataset to the notebook or pass --data-dir.")


def default_out_dir() -> Path:
    return Path("/kaggle/working/em_model") if is_kaggle() else Path("em_output")


def n_workers(requested: int | None = None) -> int:
    if requested:
        return max(1, int(requested))
    if hasattr(os, "sched_getaffinity"):
        return max(1, len(os.sched_getaffinity(0)))
    return max(1, os.cpu_count() or 1)


def _init_worker():
    for v in _THREAD_VARS:
        os.environ[v] = "1"
    try:  # workers must not handle Ctrl-C / kernel interrupts themselves
        import signal
        signal.signal(signal.SIGINT, signal.SIG_IGN)
    except Exception:
        pass


def run_pool(fn, tasks, n_jobs: int, ordered: bool = True, timeout: float = 600.0):
    """Map `fn` over `tasks` with a fork pool.

    - fork shares the big read-only arrays with workers (no pickling, no copy)
    - every result has a timeout: a stuck/killed worker (e.g. OOM-killed by
      the kernel) can no longer hang the notebook forever; remaining tasks are
      finished serially instead
    - falls back to serial execution on platforms without fork"""
    tasks = list(tasks)
    if n_jobs <= 1 or len(tasks) <= 1 or sys.platform == "win32":
        return [fn(t) for t in tasks]
    results = [None] * len(tasks)
    done = [False] * len(tasks)
    ctx = get_context("fork")
    # freeze the parent's objects out of the cyclic GC: a worker's GC pass
    # would otherwise write to every inherited container header (more COW)
    gc.collect()
    gc.freeze()
    # maxtasksperchild recycles workers: any memory a worker accumulated
    # (copy-on-write pages, caches) is returned to the OS regularly
    pool = ctx.Pool(min(n_jobs, len(tasks)), initializer=_init_worker, maxtasksperchild=25)
    try:
        asyncs = [pool.apply_async(fn, (t,)) for t in tasks]
        for i, ar in enumerate(asyncs):
            try:
                results[i] = ar.get(timeout=timeout)
                done[i] = True
            except MPTimeout:
                log(f"  worker timeout on task {i}; finishing remaining tasks serially")
                break
        pool.close()
    except BaseException:
        pool.terminate()
        raise
    finally:
        if not all(done):
            pool.terminate()
        pool.join()
        gc.unfreeze()
    for i, ok in enumerate(done):
        if not ok:
            results[i] = fn(tasks[i])
    return results


# ---------------------------------------------------------------------------
# Copy-on-write-safe strings for forked workers
# ---------------------------------------------------------------------------
# A forked worker that merely READS a Python str still writes its refcount,
# so the kernel copies every memory page it touches: with millions of
# normalised strings and 4 workers the parent's text is duplicated up to 4x
# (the classic "copy-on-read" OOM).  Workers therefore read strings from ONE
# bytes buffer + offsets (only that object's header page is ever written).
class StrCol:
    __slots__ = ("buf", "off")

    def __init__(self, values):
        enc = [("" if v is None else str(v)).encode("utf-8", "surrogatepass") for v in values]
        off = np.zeros(len(enc) + 1, np.int64)
        if enc:
            np.cumsum(np.fromiter((len(e) for e in enc), np.int64, len(enc)), out=off[1:])
        self.buf = b"".join(enc)
        self.off = array("q", off.tobytes())

    def __len__(self):
        return len(self.off) - 1

    def __getitem__(self, i):
        return self.buf[self.off[i]:self.off[i + 1]].decode("utf-8", "surrogatepass")


class CodedCol:
    """Row access to a categorical column: codes (numpy) -> StrCol of categories."""
    __slots__ = ("codes", "cats")

    def __init__(self, codes, cats):
        self.codes, self.cats = codes, cats

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, i):
        c = self.codes[i]
        return self.cats[c] if c >= 0 else ""


_STRCOL_CACHE: dict = {}


def _strcol_for(categories):
    key = id(categories)
    hit = _STRCOL_CACHE.get(key)
    if hit is None or hit[0] is not categories:
        hit = (categories, StrCol(categories))
        _STRCOL_CACHE[key] = hit
    return hit[1]


def clear_strcol_cache():
    _STRCOL_CACHE.clear()


def worker_view(df, cols):
    """dict col -> COW-safe row accessor (CodedCol for text, numpy otherwise)."""
    import pandas as pd
    out = {}
    for c in cols:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            out[c] = CodedCol(s.cat.codes.to_numpy(), _strcol_for(s.cat.categories))
        elif s.dtype == object:
            cat = s.astype("category")
            out[c] = CodedCol(cat.cat.codes.to_numpy(), StrCol(cat.cat.categories))
        else:
            out[c] = s.to_numpy()
    return out

In [ ]:
%%writefile entity_matching/data.py
"""Loading the challenge TSVs and attaching normalised columns."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .normalize import normalize_address, normalize_name
from .utils import StrCol, free, log as _log, n_workers, run_pool

USECOLS = ["entity_id", "business_name", "business_address", "country"]
_G: dict = {}

NAME_FIELDS = ["core", "full", "ocr", "skel", "nospace", "alts", "legal", "acr",
               "indic", "domain", "junk"]
ADDR_FIELDS = ["atext", "alpha", "nums", "numparts", "numstreet", "first_num",
               "state", "postal", "missing"]


def find_files(data_dir: str | Path) -> dict:
    found = {p.name: p for p in Path(data_dir).rglob("*.tsv")}
    return found


def read_tsv(path, usecols=None) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False,
                       usecols=usecols, engine="c", na_filter=False)


def load_split(data_dir, split: str):
    """Returns (s1, targets, ground_truth or None). targets = S2 + S3."""
    f = find_files(data_dir)
    s1 = read_tsv(f[f"{split}_source1.tsv"], USECOLS)
    s2 = read_tsv(f[f"{split}_source2.tsv"], USECOLS)
    s3 = read_tsv(f[f"{split}_source3.tsv"], USECOLS)
    s2["source"] = "S2"
    s3["source"] = "S3"
    tgt = pd.concat([s2, s3], ignore_index=True)
    del s2, s3
    free()
    gt = None
    if f"{split}_ground_truth.tsv" in f:
        gt = read_tsv(f[f"{split}_ground_truth.tsv"])
    for df in (s1, tgt):
        df["country"] = df["country"].astype(str).str.strip().astype("category")
    tgt["source"] = tgt["source"].astype("category")
    return s1, tgt, gt


def gt_pairs(gt: pd.DataFrame) -> pd.DataFrame:
    """Explode 'S2-1,S3-2' -> one row per (source1_entity_id, matched_id)."""
    g = gt[["source1_entity_id", "matched_entity_ids"]].copy()
    g["matched_id"] = g["matched_entity_ids"].astype(str).str.split(",")
    g = g.explode("matched_id")
    g["matched_id"] = g["matched_id"].astype(str).str.strip()
    g = g[g["matched_id"].ne("") & g["matched_id"].ne("nan")]
    return g[["source1_entity_id", "matched_id"]].reset_index(drop=True)


# ---------------------------------------------------------------------------
# Parallel normalisation over unique values (fork-shared input, no pickling)
# ---------------------------------------------------------------------------
def _name_worker(b):
    vals = _G["vals"]
    return [tuple(normalize_name(vals[i])[k] for k in NAME_FIELDS) for i in range(b[0], b[1])]


def _addr_worker(b):
    vals, ctry = _G["vals"], _G["ctry"]
    return [tuple(normalize_address(vals[i], ctry[i])[k] for k in ADDR_FIELDS)
            for i in range(b[0], b[1])]


def _run(fn, n, n_jobs, chunk=20000):
    out = []
    for res in run_pool(fn, [(i, min(i + chunk, n)) for i in range(0, n, chunk)], n_jobs):
        out.extend(res)
    return out


def _assign(df, prefix, fields, int_fields, res, codes):
    """Store each normalised field as a pandas Categorical (int32 codes + the
    unique strings once): compact, and COW-safe for workers via worker_view."""
    for j, k in enumerate(fields):
        vals = [r[j] for r in res]
        if k in int_fields:
            df[prefix + k] = np.asarray(vals, np.int8)[codes]
        else:
            fcodes, cats = pd.factorize(pd.Index(vals, dtype=object))
            df[prefix + k] = pd.Categorical.from_codes(fcodes[codes].astype(np.int32), categories=cats)
        del vals


def normalize_frame(df: pd.DataFrame, n_jobs: int | None = None, log=_log,
                    drop_raw: bool = True) -> pd.DataFrame:
    """Adds n_* (name) and a_* (address) columns.  Work is done once per
    UNIQUE value; the resulting object arrays share string objects, so
    duplicated names cost one pointer each.  Raw text is dropped afterwards."""
    n_jobs = n_workers(n_jobs)
    names = df["business_name"].astype(str)
    codes, uniq = pd.factorize(names)
    log(f"  normalising {len(uniq):,} unique names")
    _G["vals"] = StrCol(uniq)
    res = _run(_name_worker, len(uniq), n_jobs)
    _assign(df, "n_", NAME_FIELDS, ("indic", "domain", "junk"), res, codes)
    del res, codes, uniq, names

    key = pd.MultiIndex.from_arrays([df["business_address"].astype(str),
                                     df["country"].astype(str)])
    codes, uniq = pd.factorize(key)
    log(f"  normalising {len(uniq):,} unique addresses")
    _G["vals"] = StrCol(uniq.get_level_values(0))
    _G["ctry"] = StrCol(uniq.get_level_values(1))
    res = _run(_addr_worker, len(uniq), n_jobs)
    _assign(df, "a_", ADDR_FIELDS, ("missing",), res, codes)
    _G.clear()
    if drop_raw:
        df.drop(columns=["business_name", "business_address"], inplace=True)
    free()
    return df


def load_normalized(data_dir, split, cache_dir=None, n_jobs=None, s1_ids=None, log=_log):
    """load_split + normalize_frame with an optional parquet/pickle cache so a
    re-run of the notebook skips the expensive normalisation."""
    cache = Path(cache_dir) / f"norm_{split}" if cache_dir else None
    if cache is not None and (cache / "tgt.pkl").exists():
        log(f"loading normalised {split} from cache {cache}")
        s1 = pd.read_pickle(cache / "s1.pkl")
        tgt = pd.read_pickle(cache / "tgt.pkl")
        gt = pd.read_pickle(cache / "gt.pkl") if (cache / "gt.pkl").exists() else None
    else:
        log(f"loading {split} from {data_dir}")
        s1, tgt, gt = load_split(data_dir, split)
        log(f"  S1={len(s1):,} targets={len(tgt):,}")
        log("normalising S1 ...")
        normalize_frame(s1, n_jobs, log)
        log("normalising targets ...")
        normalize_frame(tgt, n_jobs, log)
        if cache is not None:
            cache.mkdir(parents=True, exist_ok=True)
            s1.to_pickle(cache / "s1.pkl")
            tgt.to_pickle(cache / "tgt.pkl")
            if gt is not None:
                gt.to_pickle(cache / "gt.pkl")
    if s1_ids is not None:
        s1 = s1[s1["entity_id"].isin(s1_ids)].reset_index(drop=True)
    return s1, tgt, gt

In [ ]:
%%writefile entity_matching/blocking.py
"""
Candidate generation (blocking).

Country is a HARD block (cell 9: 100 % of true links are same-country).
Inside a country every record emits hashed keys of several types; an S1
record is paired with every target sharing one of its rarest keys and the
pair is scored by the IDF-weighted sum of shared keys.  We keep the top-K
pairs per S1 by total score plus the top-K2 by name-only and address-only
score, so a record whose name is unrecognisable (cell 17: names replaced by
"Kelopyrahalo") can still be found through its address, and vice versa.

EDA motivation: name token OR number gives 96.9 % recall and adding rare
address tokens 99.7 % (cells 21-23) - but with capped fan-out.
"""
from __future__ import annotations

from dataclasses import dataclass, field
import numpy as np
import pandas as pd

from .normalize import skeleton
from .utils import free, run_pool, worker_view

_G: dict = {}

# key type -> (weight, max document frequency, max keys per record, group)
# group: 0 = name evidence, 1 = address evidence
KEY_TYPES = {
    "N": (1.0, 2000, 6, 0),   # skeleton name token
    "B": (1.6, 2000, 4, 0),   # skeleton name bigram
    "P": (1.6, 1000, 2, 0),   # first 8 chars of space-less core (domains, spacing)
    "F": (3.0, 2000, 1, 0),   # full skeleton name
    "A": (0.6, 3000, 6, 1),   # address word
    "U": (1.0, 3000, 6, 1),   # address number (canonical)
    "S": (2.2, 1000, 4, 1),   # number + following street word
    "C": (2.5, 1000, 6, 1),   # name token x address number ("arihant|417")
}
TYPE_CODE = {k: i for i, k in enumerate(KEY_TYPES)}


@dataclass
class BlockConfig:
    top_k: int = 60          # k_max of the adaptive forward list
    k_min: int = 12          # always keep this many per S1
    adapt_ratio: float = 0.25  # beyond k_min keep only score >= ratio * top1 (SABER-style)
    top_k_name: int = 10
    top_k_addr: int = 10
    reverse_k: int = 5       # bidirectional: each target keeps its top-R S1 records
    posting_budget: int = 2500  # max target postings expanded per S1 record
    min_keys: int = 6        # ... but always use the 6 rarest keys
    pair_budget: int = 5_000_000  # max expanded postings per chunk (~0.6 GB temp)
    max_df_frac: float = 0.01
    n_jobs: int = 4
    key_types: dict = field(default_factory=lambda: dict(KEY_TYPES))


def _keys_for(skel, alts, nospace, alpha, nums, numstreet):
    out = []
    toks = [t for t in skel.split() if len(t) >= 2]
    for t in dict.fromkeys(toks):
        out.append(("N", t))
    for a, b in zip(toks, toks[1:]):
        out.append(("B", a + "_" + b))
    if alts:
        for alt in alts.split("||"):
            for t in alt.split():
                s = skeleton(t)
                if len(s) >= 2:
                    out.append(("N", s))
            ns = alt.replace(" ", "")
            if len(ns) >= 5:
                out.append(("P", ns[:8]))
    if len(nospace) >= 5:
        out.append(("P", nospace[:8]))
    if skel:
        out.append(("F", skel.replace(" ", "")))
    for t in dict.fromkeys(alpha.split()):
        if len(t) >= 3:
            out.append(("A", t))
    for t in dict.fromkeys(nums.split()):
        out.append(("U", t))
    for t in dict.fromkeys(numstreet.split()):
        out.append(("S", t))
    # composite keys stay rare even when the name word and the number are
    # both common (generic names such as "pediatric group", EDA cell 20)
    # 2 longest (≈ rarest) name tokens x first 3 numbers = at most 6 keys
    num_list = list(dict.fromkeys(nums.split()))[:3]
    for t in sorted(dict.fromkeys(toks), key=len, reverse=True)[:2]:
        for n in num_list:
            out.append(("C", t + "|" + n))
    return out


def _key_worker(b):
    lo, hi = b
    cols = _G["cols"]
    idx, codes, strs = [], [], []
    for i in range(hi - lo):
        row = tuple(c[lo + i] for c in cols)
        seen = set()
        for typ, val in _keys_for(*row):
            k = typ + ":" + val
            if k in seen:  # de-duplicate per record here: no pandas pass later
                continue
            seen.add(k)
            idx.append(lo + i)
            codes.append(TYPE_CODE[typ])
            strs.append(k)
    h = pd.util.hash_array(np.array(strs, dtype=object)) if strs else np.zeros(0, np.uint64)
    return (np.asarray(idx, np.int32), np.asarray(codes, np.int8), h)


def build_keys(df: pd.DataFrame, n_jobs: int = 4, chunk: int = 50000):
    """Hashed blocking keys -> (record index int32, key type int8, hash uint64)."""
    cols_names = ["n_skel", "n_alts", "n_nospace", "a_alpha", "a_nums", "a_numstreet"]
    view = worker_view(df, cols_names)  # COW-safe, fork-shared
    _G["cols"] = [view[c] for c in cols_names]
    res = run_pool(_key_worker, [(s, min(s + chunk, len(df))) for s in range(0, len(df), chunk)], n_jobs)
    _G.clear()
    if not res:
        return np.zeros(0, np.int32), np.zeros(0, np.int8), np.zeros(0, np.uint64)
    out = (np.concatenate([r[0] for r in res]),
           np.concatenate([r[1] for r in res]),
           np.concatenate([r[2] for r in res]))
    del res
    return out


def _topk_mask(s, score, k):
    """Boolean mask of the top-k rows by score within each s (s sorted not required)."""
    order = np.lexsort((-score, s))
    s_sorted = s[order]
    first = np.r_[True, s_sorted[1:] != s_sorted[:-1]]
    grp_start = np.maximum.accumulate(np.where(first, np.arange(len(s_sorted)), 0))
    rank = np.arange(len(s_sorted)) - grp_start
    mask = np.zeros(len(s), bool)
    mask[order[rank < k]] = True
    return mask


def generate_candidates(s1: pd.DataFrame, tgt: pd.DataFrame, cfg: BlockConfig,
                        s1_rows: np.ndarray | None = None, log=print) -> pd.DataFrame:
    """Returns DataFrame[s_row, t_row, blk_score, blk_name, blk_addr, blk_nkeys]."""
    kt = cfg.key_types
    weights = np.array([v[0] for v in kt.values()])
    caps = np.array([v[1] for v in kt.values()])
    per_rec = np.array([v[2] for v in kt.values()])
    group = np.array([v[3] for v in kt.values()])

    if s1_rows is None:
        s1_rows = np.arange(len(s1))
    out, rev_pool = [], []
    s_ctry = s1["country"].astype(str).to_numpy()
    t_ctry = tgt["country"].astype(str).to_numpy()
    for country in sorted(pd.unique(s_ctry[s1_rows])):
        s_rows = s1_rows[s_ctry[s1_rows] == country]
        t_rows = np.flatnonzero(t_ctry == country)
        if len(s_rows) == 0 or len(t_rows) == 0:
            continue
        log(f"  [{country}] S1={len(s_rows):,} targets={len(t_rows):,}")
        t_idx, t_typ, t_h = build_keys(tgt.iloc[t_rows], cfg.n_jobs)
        # one sort gives document frequencies AND posting lists (no pandas,
        # no np.unique inverse: peak RAM ~ 3 x 8 bytes per key)
        order = np.argsort(t_h, kind="stable")
        post_h = t_h[order]
        del t_h
        post_t = t_idx[order]
        del t_idx
        typ_sorted = t_typ[order]
        del t_typ, order
        free()
        brk = np.flatnonzero(np.r_[True, post_h[1:] != post_h[:-1]])
        klen = np.diff(np.r_[brk, len(post_h)])
        # caps are absolute for the full data and shrink with small target sets
        eff_caps = np.maximum(20, np.minimum(caps, cfg.max_df_frac * len(t_rows))).astype(np.int64)
        keep_key = klen <= eff_caps[typ_sorted[brk]]
        del typ_sorted
        keep_el = np.repeat(keep_key, klen)
        post_h, post_t = post_h[keep_el], post_t[keep_el]
        del keep_el
        ph_len = klen[keep_key]
        ph_start = np.r_[0, np.cumsum(ph_len)[:-1]] if len(ph_len) else np.zeros(0, np.int64)
        ph_uniq = post_h[ph_start] if len(ph_len) else np.zeros(0, np.uint64)
        del post_h, brk, klen, keep_key
        free()
        n_t = len(t_rows)

        if len(ph_uniq) == 0:
            continue
        s_idx, s_typ, s_h = build_keys(s1.iloc[s_rows], cfg.n_jobs)
        pos = np.clip(np.searchsorted(ph_uniq, s_h), 0, len(ph_uniq) - 1)
        found = ph_uniq[pos] == s_h
        s_idx, s_typ, pos = s_idx[found], s_typ[found], pos[found]
        del s_h, found
        dfk = ph_len[pos]
        # keep the per_rec rarest keys of each type per S1 record
        o = np.lexsort((dfk, s_typ, s_idx))
        s_idx, s_typ, pos, dfk = s_idx[o], s_typ[o], pos[o], dfk[o]
        grp = np.r_[True, (s_idx[1:] != s_idx[:-1]) | (s_typ[1:] != s_typ[:-1])]
        gstart = np.maximum.accumulate(np.where(grp, np.arange(len(grp)), 0))
        keep = (np.arange(len(grp)) - gstart) < per_rec[s_typ]
        s_idx, s_typ, pos, dfk = s_idx[keep], s_typ[keep], pos[keep], dfk[keep]
        # per-S1 posting budget: rarest keys first, stop once ~budget target
        # postings are covered (always keep the min_keys rarest) -> bounded time
        o = np.lexsort((dfk, s_idx))
        s_idx, s_typ, pos, dfk = s_idx[o], s_typ[o], pos[o], dfk[o]
        first = np.r_[True, s_idx[1:] != s_idx[:-1]] if len(s_idx) else np.zeros(0, bool)
        gstart = np.maximum.accumulate(np.where(first, np.arange(len(s_idx)), 0))
        cum = np.cumsum(dfk)
        before = cum - dfk - (cum[gstart] - dfk[gstart])
        rank = np.arange(len(s_idx)) - gstart
        keep = (rank < cfg.min_keys) | (before < cfg.posting_budget)
        sk = pd.DataFrame({"i": s_idx[keep], "p": pos[keep], "c": s_typ[keep], "df": dfk[keep]})
        del s_idx, s_typ, pos, dfk, o, grp, gstart, keep, first, cum, before, rank
        sk["w"] = weights[sk["c"].to_numpy()] * np.log1p(n_t / sk["df"].to_numpy())
        sk["g"] = group[sk["c"].to_numpy()]

        # chunk S1 records so each chunk expands to <= pair_budget postings:
        # RAM stays bounded whatever the key frequencies are
        si_all = sk["i"].to_numpy()
        cum = np.cumsum(ph_len[sk["p"].to_numpy()])
        rec_start = np.flatnonzero(np.r_[True, si_all[1:] != si_all[:-1]])
        rec_cum = np.r_[0, cum][rec_start]
        cuts = rec_start[np.unique(np.searchsorted(
            rec_cum, np.arange(0, (cum[-1] if len(cum) else 0) + cfg.pair_budget, cfg.pair_budget)))
            .clip(0, len(rec_start) - 1)] if len(rec_start) else np.zeros(0, np.int64)
        bounds = np.unique(np.r_[cuts, len(sk)])
        bounds = np.r_[0, bounds[bounds > 0]] if len(bounds) else np.zeros(1, np.int64)
        for b0, b1 in zip(bounds[:-1], bounds[1:]):
            if b1 <= b0:
                continue
            ch = sk.iloc[b0:b1]
            starts = ph_start[ch["p"].to_numpy()]
            lens = ph_len[ch["p"].to_numpy()]
            total = int(lens.sum())
            if total == 0:
                continue
            rep = np.repeat(np.arange(len(ch)), lens)
            offs = np.arange(total) - np.repeat(np.cumsum(lens) - lens, lens)
            tt = post_t[np.repeat(starts, lens) + offs]
            ss = ch["i"].to_numpy()[rep]
            w = ch["w"].to_numpy()[rep]
            g = ch["g"].to_numpy()[rep]
            pair = ss.astype(np.int64) * (n_t + 1) + tt
            o = np.argsort(pair, kind="quicksort")
            pair_s = pair[o]
            brk = np.flatnonzero(np.r_[True, pair_s[1:] != pair_s[:-1]])
            up = pair_s[brk]
            wo, go = w[o], g[o]
            score = np.add.reduceat(wo, brk)
            sname = np.add.reduceat(wo * (go == 0), brk)
            saddr = score - sname
            nkeys = np.diff(np.r_[brk, len(pair_s)])
            del o, pair_s, wo, go, pair, ss, tt, w, g, rep, offs
            ps = up // (n_t + 1)
            pt = up % (n_t + 1)
            # forward, adaptive K: rank < k_min OR (rank < k_max AND score >= ratio*top1)
            rank, top1 = _rank_top1(ps, score)
            fwd = (rank < cfg.k_min) | ((rank < cfg.top_k) & (score >= cfg.adapt_ratio * top1))
            m = (fwd | (_rank_top1(ps, sname)[0] < cfg.top_k_name)
                 | (_rank_top1(ps, saddr)[0] < cfg.top_k_addr))
            # reverse direction (target -> S1): chunk-local top-R is a superset
            # of the global top-R, merged and re-pruned below
            rev = (_rank_top1(pt, score)[0] < cfg.reverse_k) if cfg.reverse_k else np.zeros(len(ps), bool)
            keep = m | rev
            frame = pd.DataFrame({
                "s_row": s_rows[ps[keep]].astype(np.int32), "t_row": t_rows[pt[keep]].astype(np.int32),
                "blk_score": score[keep].astype(np.float32),
                "blk_name": sname[keep].astype(np.float32),
                "blk_addr": saddr[keep].astype(np.float32),
                "blk_nkeys": nkeys[keep].astype(np.int16),
            })
            out.append(frame[m[keep]])
            if cfg.reverse_k:
                rev_pool.append(frame[rev[keep]])
                if sum(len(r) for r in rev_pool) > 4 * cfg.reverse_k * n_t:
                    rev_pool = [_prune_reverse(pd.concat(rev_pool, ignore_index=True), cfg.reverse_k)]
        if cfg.reverse_k and rev_pool:
            out.append(_prune_reverse(pd.concat(rev_pool, ignore_index=True), cfg.reverse_k))
        rev_pool = []
        del post_t, ph_uniq, ph_start, ph_len, sk
        free()
    if not out:
        return pd.DataFrame(columns=["s_row", "t_row", "blk_score", "blk_name", "blk_addr", "blk_nkeys"])
    res = pd.concat(out, ignore_index=True).drop_duplicates(["s_row", "t_row"])
    return res.reset_index(drop=True)


def _rank_top1(g, v):
    """Rank of each row within its group g by descending v, and the group max.
    One float argsort on a composite key (much faster than lexsort)."""
    n = len(v)
    if n == 0:
        return np.zeros(0, np.int64), np.zeros(0)
    v = np.asarray(v, np.float64)
    vmax = float(v.max()) + 1.0
    scale = 2.0 ** np.ceil(np.log2(vmax + 1.0))
    comp = g.astype(np.float64) * scale + (vmax - v)
    o = np.argsort(comp, kind="quicksort")
    gs = g[o]
    first = np.r_[True, gs[1:] != gs[:-1]]
    start = np.maximum.accumulate(np.where(first, np.arange(n), 0))
    rank = np.empty(n, np.int64)
    rank[o] = np.arange(n) - start
    top = np.empty(n)
    top[o] = v[o][start]
    return rank, top


def _prune_reverse(df, k):
    """Keep each target's top-k S1 records by blocking score."""
    df = df.sort_values(["t_row", "blk_score"], ascending=[True, False])
    return df[df.groupby("t_row").cumcount().to_numpy() < k]

In [ ]:
%%writefile entity_matching/features.py
"""
Pairwise comparison features for (S1, target) candidate pairs.

Groups
------
name      : fuzzy ratios on several normalised views (core / OCR-folded /
            phonetic skeleton / space-less), IDF-weighted token overlap,
            alias (DBA) best match, domain containment, acronym, legal form.
address   : fuzzy ratios, IDF-weighted word overlap, number agreement
            (exact / compound / parts / house number / number+street),
            state agreement, missing flags.
rules     : hard-coded boolean patterns (exact core + shared number, ...)
context   : rank / gap of the pair among the S1's candidates and among the
            S1 records competing for the same target (one-owner, cell 8).
"""
from __future__ import annotations

import math
from collections import Counter
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

from .utils import run_pool, worker_view

PAIR_FEATURES = [
    # name
    "n_ratio_full", "n_ratio_core", "n_tsort_core", "n_tset_core", "n_partial_core",
    "n_wratio_core", "n_jw_nospace", "n_ratio_ocr", "n_tset_ocr", "n_ratio_skel",
    "n_tset_skel", "n_idf_jacc", "n_idf_contain", "n_idf_shared_max",
    "n_idf_miss_s", "n_idf_miss_t", "n_alt_best", "n_nospace_contain",
    "n_nospace_partial", "n_acr", "n_legal_both", "n_legal_eq", "n_legal_conflict",
    "n_ntok_s", "n_ntok_t", "n_ntok_diff", "n_exact_core", "n_exact_skel",
    "n_core_df_t", "s_indic", "t_indic", "t_domain", "t_junk",
    # address
    "a_missing_t", "a_ratio", "a_tset", "a_tsort", "a_partial", "a_idf_jacc",
    "a_idf_contain", "a_ratio_alpha_nospace", "num_s", "num_t", "num_shared",
    "num_jacc", "num_t_only", "num_s_only", "num_compound_shared", "numparts_shared",
    "numparts_jacc", "first_num_eq", "numstreet_shared", "state_eq", "postal_eq",
    "num_idf_shared", "num_idf_max", "num_idf_t_only", "first_num_conflict",
    # rules
    "r_exact_core_num", "r_skel_num", "r_domain_num", "r_alias_num",
    "r_name_only_strong", "r_addr_only_strong",
    # misc
    "is_s3",
]
BLOCK_FEATURES = ["blk_score", "blk_name", "blk_addr", "blk_nkeys"]
CONTEXT_FEATURES = [
    "ctx_ncand_s", "ctx_rank_s", "ctx_gap_s", "ctx_ncand_t", "ctx_rank_t", "ctx_gap_t",
    "ctx_blk_rank_s", "ctx_same_core_in_s", "ctx_same_num_in_s",
]
ALL_FEATURES = PAIR_FEATURES + BLOCK_FEATURES + CONTEXT_FEATURES

_LEGAL_EQUIV = [{"llc", "pllc", "lc"}, {"inc", "corp"}, {"ltd", "pvt", "public", "plc"},
                {"llp", "lp"}, {"pc", "pa"}, {"sarl", "eurl"}, {"sas", "sasu", "sa"}]

# globals shared with forked workers
_G: dict = {}


def build_idf(tgt: pd.DataFrame) -> dict:
    """Per-country IDF tables from TARGET records (never from S1/test labels)."""
    idf = {}
    for country, grp in tgt.groupby(tgt["country"].astype(str), observed=True):
        n = len(grp)
        cn, ca, cu = Counter(), Counter(), Counter()
        for s in grp["n_skel"]:
            cn.update(set(s.split()))
        for s in grp["a_alpha"]:
            ca.update(set(s.split()))
        for s in grp["a_nums"]:
            cu.update(set(s.split()))
        core_df = Counter(grp["n_skel"])
        idf[country] = {
            "n": {k: math.log1p(n / v) for k, v in cn.items()},
            "a": {k: math.log1p(n / v) for k, v in ca.items()},
            "u": {k: math.log1p(n / v) for k, v in cu.items()},
            "core_df": dict(core_df),
            "default": math.log1p(n),
        }
    return idf


def _idf_overlap(a_toks, b_toks, table, default):
    if not a_toks or not b_toks:
        return 0.0, 0.0, 0.0, 1.0, 1.0
    sa, sb = set(a_toks), set(b_toks)
    w = lambda t: table.get(t, default)  # noqa: E731
    inter = sa & sb
    wi = sum(w(t) for t in inter)
    wa = sum(w(t) for t in sa)
    wb = sum(w(t) for t in sb)
    wu = wa + wb - wi
    return (wi / wu if wu else 0.0,
            wi / min(wa, wb) if min(wa, wb) else 0.0,
            max((w(t) for t in inter), default=0.0),
            (wa - wi) / wa if wa else 1.0,
            (wb - wi) / wb if wb else 1.0)


def _legal_rel(a, b):
    sa, sb = set(a.split()), set(b.split())
    both = int(bool(sa) and bool(sb))
    eq = int(both and sa == sb)
    conflict = 0
    if both and not (sa & sb):
        conflict = 1
        for grp in _LEGAL_EQUIV:
            if sa & grp and sb & grp:
                conflict = 0
    return both, eq, conflict


def _pair(i, j):
    S, T = _G["S"], _G["T"]
    country = str(S["country"][i])
    tab = _G["idf"].get(country) or {"n": {}, "a": {}, "u": {}, "core_df": {}, "default": 10.0}
    d = tab["default"]

    sc, tc = S["n_core"][i], T["n_core"][j]
    sf, tf = S["n_full"][i], T["n_full"][j]
    so, to = S["n_ocr"][i], T["n_ocr"][j]
    ss, ts = S["n_skel"][i], T["n_skel"][j]
    sn, tn = S["n_nospace"][i], T["n_nospace"][j]
    stoks, ttoks = ss.split(), ts.split()

    jacc, contain, smax, miss_s, miss_t = _idf_overlap(stoks, ttoks, tab["n"], d)

    alt_best = 0.0
    for alt in (T["n_alts"][j].split("||") if T["n_alts"][j] else []):
        alt_best = max(alt_best, fuzz.token_set_ratio(sc, alt))
    for alt in (S["n_alts"][i].split("||") if S["n_alts"][i] else []):
        alt_best = max(alt_best, fuzz.token_set_ratio(alt, tc))

    short, long_ = (sn, tn) if len(sn) <= len(tn) else (tn, sn)
    nospace_contain = int(len(short) >= 4 and short in long_)
    acr = int(bool(S["n_acr"][i] and S["n_acr"][i] == tn) or bool(T["n_acr"][j] and T["n_acr"][j] == sn))
    lboth, leq, lconf = _legal_rel(S["n_legal"][i], T["n_legal"][j])

    # address
    sa_txt, ta_txt = S["a_atext"][i], T["a_atext"][j]
    salpha, talpha = S["a_alpha"][i].split(), T["a_alpha"][j].split()
    ajacc, acontain, _, _, _ = _idf_overlap(salpha, talpha, tab["a"], d)
    snum, tnum = set(S["a_nums"][i].split()), set(T["a_nums"][j].split())
    shared = snum & tnum
    union = snum | tnum
    comp_shared = sum(1 for x in shared if ("/" in x or "-" in x or not x.isdigit()))
    sp, tp = set(S["a_numparts"][i].split()), set(T["a_numparts"][j].split())
    sp_sh = len(sp & tp)
    sp_un = len(sp | tp)
    sns, tns = set(S["a_numstreet"][i].split()), set(T["a_numstreet"][j].split())
    sst, tst = S["a_state"][i], T["a_state"][j]
    state_eq = (1 if set(sst.split()) & set(tst.split()) else -1) if sst and tst else 0
    spo, tpo = S["a_postal"][i], T["a_postal"][j]
    postal_eq = (1 if set(spo.split()) & set(tpo.split()) else -1) if spo and tpo else 0
    a_missing = int(T["a_missing"][j])

    n_tset_skel = fuzz.token_set_ratio(ss, ts)
    a_tset = fuzz.token_set_ratio(sa_txt, ta_txt) if not a_missing else -1.0
    exact_core = int(sc == tc and sc != "")
    exact_skel = int(ss == ts and ss != "")
    n_shared = len(shared)
    # rarity-weighted number evidence: sharing "63/1/2" >> sharing "12"
    ut = tab["u"]
    num_idf = [ut.get(x, d) for x in shared]
    num_idf_t_only = sum(ut.get(x, d) for x in (tnum - snum))
    f_s, f_t = S["a_first_num"][i], T["a_first_num"][j]
    first_conflict = int(bool(f_s) and bool(f_t) and f_s != f_t and f_t not in snum and f_s not in tnum)

    return (
        fuzz.ratio(sf, tf), fuzz.ratio(sc, tc), fuzz.token_sort_ratio(sc, tc),
        fuzz.token_set_ratio(sc, tc), fuzz.partial_ratio(sc, tc), fuzz.WRatio(sc, tc),
        JaroWinkler.normalized_similarity(sn, tn) * 100, fuzz.ratio(so, to),
        fuzz.token_set_ratio(so, to), fuzz.ratio(ss, ts), n_tset_skel,
        jacc, contain, smax, miss_s, miss_t, alt_best, nospace_contain,
        fuzz.partial_ratio(sn, tn), acr, lboth, leq, lconf,
        len(stoks), len(ttoks), abs(len(stoks) - len(ttoks)), exact_core, exact_skel,
        math.log1p(tab["core_df"].get(ts, 0)), S["n_indic"][i], T["n_indic"][j],
        T["n_domain"][j], T["n_junk"][j],
        a_missing,
        fuzz.ratio(sa_txt, ta_txt) if not a_missing else -1.0,
        a_tset,
        fuzz.token_sort_ratio(sa_txt, ta_txt) if not a_missing else -1.0,
        fuzz.partial_ratio(sa_txt, ta_txt) if not a_missing else -1.0,
        ajacc, acontain,
        fuzz.ratio("".join(sorted(salpha)), "".join(sorted(talpha))) if not a_missing else -1.0,
        len(snum), len(tnum), n_shared, n_shared / len(union) if union else 0.0,
        len(tnum - snum), len(snum - tnum), comp_shared, sp_sh, sp_sh / sp_un if sp_un else 0.0,
        int(bool(S["a_first_num"][i]) and S["a_first_num"][i] == T["a_first_num"][j]),
        len(sns & tns), state_eq, postal_eq,
        sum(num_idf), max(num_idf, default=0.0), num_idf_t_only, first_conflict,
        # rules
        int(exact_core and n_shared > 0), int(exact_skel and n_shared > 0),
        int(T["n_domain"][j] and nospace_contain and n_shared > 0),
        int(alt_best >= 95 and n_shared > 0),
        int(n_tset_skel >= 95 and contain >= 0.9),
        int((a_tset >= 90) and n_shared >= 2),
        int(str(T["source"][j]) == "S3"),
    )


def _worker(bounds):
    lo, hi = bounds
    si, ti = _G["si"], _G["ti"]
    out = np.empty((hi - lo, len(PAIR_FEATURES)), np.float32)
    for r in range(lo, hi):
        out[r - lo] = _pair(si[r], ti[r])
    return lo, out


_S_COLS = ["country", "n_core", "n_full", "n_ocr", "n_skel", "n_nospace", "n_alts",
           "n_acr", "n_legal", "n_indic", "a_atext", "a_alpha", "a_nums", "a_numparts",
           "a_numstreet", "a_state", "a_postal", "a_first_num", "a_missing"]
_T_COLS = _S_COLS + ["n_domain", "n_junk", "source"]


def compute_pair_features(cands: pd.DataFrame, s1: pd.DataFrame, tgt: pd.DataFrame,
                          idf: dict, n_jobs: int = 4, chunk: int = 20000) -> pd.DataFrame:
    # zero-copy numpy views, shared with forked workers (nothing is pickled)
    _G["S"] = worker_view(s1, _S_COLS)   # COW-safe string access in workers
    _G["T"] = worker_view(tgt, _T_COLS)
    _G["idf"] = idf
    _G["si"] = cands["s_row"].to_numpy()
    _G["ti"] = cands["t_row"].to_numpy()
    n = len(cands)
    X = np.empty((n, len(PAIR_FEATURES)), np.float32)
    for lo, arr in run_pool(_worker, [(lo, min(lo + chunk, n)) for lo in range(0, n, chunk)], n_jobs):
        X[lo:lo + len(arr)] = arr
    _G.clear()
    feats = pd.DataFrame(X, columns=PAIR_FEATURES, index=cands.index)
    return pd.concat([cands, feats], axis=1)


def add_target_context(cands: pd.DataFrame) -> pd.DataFrame:
    """Competition for the same target across ALL S1 records (one-owner
    constraint).  Uses blocking scores so it can be computed globally before
    the batched feature pass."""
    g = cands.groupby("t_row")["blk_score"]
    cands["ctx_ncand_t"] = g.transform("size").astype(np.float32)
    cands["ctx_rank_t"] = g.rank(ascending=False, method="min").astype(np.float32)
    cands["ctx_gap_t"] = (g.transform("max") - cands["blk_score"]).astype(np.float32)
    return cands


def add_s1_context(df: pd.DataFrame, tgt: pd.DataFrame) -> pd.DataFrame:
    """Ranking features within each S1's candidate list (needs all candidates
    of an S1 in the same batch)."""
    comb = (df["n_tset_skel"].to_numpy() + df["n_idf_contain"].to_numpy() * 100
            + np.clip(df["a_tset"].to_numpy(), 0, None) + 20 * df["num_jacc"].to_numpy())
    df["_comb"] = comb
    gs = df.groupby("s_row")["_comb"]
    df["ctx_ncand_s"] = gs.transform("size").astype(np.float32)
    df["ctx_rank_s"] = gs.rank(ascending=False, method="min").astype(np.float32)
    df["ctx_gap_s"] = (gs.transform("max") - comb).astype(np.float32)
    df["ctx_blk_rank_s"] = df.groupby("s_row")["blk_score"].rank(
        ascending=False, method="min").astype(np.float32)
    # siblings: an S1 usually has several S2/S3 duplicates of itself
    # group on integer codes (categorical columns) - no string materialisation
    tr = df["t_row"].to_numpy()
    df["_core"] = _codes(tgt["n_skel"])[tr]
    fcol = tgt["a_first_num"]
    df["_fnum"] = _codes(fcol)[tr]
    df["ctx_same_core_in_s"] = df.groupby(["s_row", "_core"])["_comb"].transform("size").astype(np.float32)
    df["ctx_same_num_in_s"] = df.groupby(["s_row", "_fnum"])["_comb"].transform("size").astype(np.float32)
    df.loc[df["_fnum"] == _empty_code(fcol), "ctx_same_num_in_s"] = 0
    return df.drop(columns=["_comb", "_core", "_fnum"])


def _codes(col):
    if isinstance(col.dtype, pd.CategoricalDtype):
        return col.cat.codes.to_numpy()
    return pd.factorize(col)[0]


def _empty_code(col):
    if isinstance(col.dtype, pd.CategoricalDtype):
        cats = col.cat.categories
        return int(cats.get_loc("")) if "" in cats else -99
    codes, uniq = pd.factorize(col)
    return int(np.flatnonzero(uniq == "")[0]) if (uniq == "").any() else -99

In [ ]:
%%writefile entity_matching/decision.py
"""
Decision layer: calibration + per-S1 expected-F0.5 set selection.

The official metric (per public write-ups) is F0.5 computed per Source-1
entity and averaged (macro), with an empty prediction for an S1 without
true matches scoring 1.  For one S1 with calibrated, one-owner-resolved
candidate probabilities q1 >= q2 >= ..., predicting the top-k set has

    E[F_b | top-k] ~= (1+b^2) * sum_{i<=k} q_i / (b^2 * E|T| + k)
    E[F_b | empty] =  P(T empty) ~= prod_i (1 - q_i) * exp(-lam)

with E|T| = sum_i q_i + lam, lam = expected true links missed by blocking.
We pick the k (0 included) with the largest value - the plug-in version of
the General F-measure Maximizer (Dembczynski et al., NIPS 2011; Waegeman
et al., JMLR 2014).  `empty_bias` scales E[F|empty] and is tuned on OOF.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression


class Calibrator:
    """Isotonic calibration stored as monotone knots (np.interp at predict)."""

    def __init__(self, x=None, y=None):
        self.x, self.y = x, y

    def fit(self, p, y):
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
        iso.fit(np.asarray(p, float), np.asarray(y, float))
        self.x, self.y = iso.X_thresholds_, iso.y_thresholds_
        return self

    def __call__(self, p):
        return np.interp(np.asarray(p, float), self.x, self.y).astype(np.float32)

    def to_dict(self):
        return {"x": np.asarray(self.x).tolist(), "y": np.asarray(self.y).tolist()}

    @staticmethod
    def from_dict(d):
        return Calibrator(np.asarray(d["x"]), np.asarray(d["y"]))


def one_owner(t_row, q):
    """Mask keeping, for every target, only its most probable S1."""
    order = np.lexsort((-q, t_row))
    keep = np.zeros(len(q), bool)
    ts = t_row[order]
    keep[order[np.r_[True, ts[1:] != ts[:-1]]]] = True
    return keep


def expected_f_select(s_row, t_row, q, beta=0.5, lam=0.0, empty_bias=1.0,
                      min_q=0.0, n_s=None) -> np.ndarray:
    """Boolean mask of selected pairs maximising per-S1 expected F_beta."""
    b2 = beta * beta
    n = len(q)
    owned = one_owner(t_row, q)
    # expected truth size uses ALL candidates of the S1 (owned or not)
    exp_t = pd.Series(q).groupby(s_row).transform("sum").to_numpy() + lam
    log_empty = pd.Series(np.log1p(-np.clip(q, 0, 1 - 1e-7))).groupby(s_row).transform("sum").to_numpy()
    p_empty = np.exp(log_empty - lam) * empty_bias

    idx = np.flatnonzero(owned & (q > min_q))
    sel = np.zeros(n, bool)
    if not len(idx):
        return sel
    d = pd.DataFrame({"s": s_row[idx], "q": q[idx], "i": idx,
                      "et": exp_t[idx], "pe": p_empty[idx]}).sort_values(["s", "q"], ascending=[True, False])
    d["k"] = d.groupby("s").cumcount() + 1
    d["cum"] = d.groupby("s")["q"].cumsum()
    d["ef"] = (1 + b2) * d["cum"] / (b2 * d["et"] + d["k"])
    best = d.groupby("s")["ef"].transform("max")
    kbest = d["k"].where(d["ef"] == best).groupby(d["s"]).transform("min")
    take = (d["k"] <= kbest) & (best > d["pe"])
    sel[d["i"].to_numpy()[take.to_numpy()]] = True
    return sel


def threshold_select(s_row, t_row, q, thr, alpha=0.0):
    """Baseline rule: q >= thr, one owner per target, q >= alpha * max q of S1."""
    mask = (q >= thr) & one_owner(t_row, q)
    if alpha > 0 and mask.any():
        idx = np.flatnonzero(mask)
        mx = pd.Series(q[idx]).groupby(s_row[idx]).transform("max").to_numpy()
        mask[idx[q[idx] < alpha * mx]] = False
    return mask

In [ ]:
%%writefile entity_matching/stage2.py
"""
Stage 2: graph / cluster features built from stage-1 probabilities.

An S1 entity is usually matched by several S2/S3 duplicates of itself
(EDA cell 7: 3.46 links per S1 on average), and every target belongs to at
most one S1 (cell 8).  Stage 2 turns those structural facts into features:

sibling   : how similar is candidate t to the most probable OTHER candidates
            of the same S1?  A weak candidate that looks like a confident
            sibling is probably a duplicate of it (label propagation over the
            candidate graph, cf. multi-source clustering / TransClean).
s-context : rank, gap, mass of stage-1 probabilities inside the S1's list
t-context : strongest competing S1 for the same target (one-owner)
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from rapidfuzz import fuzz

from .utils import run_pool, worker_view

CARRY = ["n_tset_skel", "n_idf_contain", "n_alt_best", "n_exact_skel", "a_tset",
         "num_shared", "num_jacc", "a_missing_t", "n_core_df_t", "blk_score",
         "ctx_ncand_s", "t_indic", "t_domain", "is_s3"]
STAGE2_FEATURES = CARRY + [
    "p1", "p1_logit", "p1_rank_s", "p1_gap_s", "p1_second_s", "p1_mass_other_s",
    "p1_n50_s", "p1_other_max_t", "p1_gap_t", "p1_n10_t",
    "sib_name_max", "sib_addr_max", "sib_num_max", "sib_score", "sib_p1_best",
]
_G: dict = {}
SIB_TOP = 3        # compare each candidate with the 3 most probable others
SIB_MIN_P1 = 0.02  # candidates below this are hopeless - skip the fuzz work


def _sib_worker(b):
    lo, hi = b
    ta, tb = _G["ta"], _G["tb"]
    skel, atext, nums = _G["skel"], _G["atext"], _G["nums"]
    out = np.zeros((hi - lo, 3), np.float32)
    for r in range(lo, hi):
        i, j = ta[r], tb[r]
        out[r - lo, 0] = fuzz.token_set_ratio(skel[i], skel[j])
        out[r - lo, 1] = fuzz.token_set_ratio(atext[i], atext[j]) if atext[i] and atext[j] else 0.0
        ni, nj = set(nums[i].split()), set(nums[j].split())
        out[r - lo, 2] = len(ni & nj) / len(ni | nj) if (ni and nj) else 0.0
    return lo, out


def top2_by_group(keys, vals):
    """Per row: (max of its group, max of its group excluding this row)."""
    n = len(vals)
    order = np.lexsort((-vals, keys))
    ks, vs = keys[order], vals[order]
    first = np.r_[True, ks[1:] != ks[:-1]] if n else np.zeros(0, bool)
    start = np.maximum.accumulate(np.where(first, np.arange(n), 0))
    rank = np.arange(n) - start
    top1 = vs[start]
    same_next = np.r_[ks[1:] == ks[:-1], False]
    second_at = np.where(same_next, np.r_[vs[1:], 0.0], 0.0)[start]
    other = np.where(rank == 0, second_at, top1)
    out1, out2 = np.empty(n, np.float32), np.empty(n, np.float32)
    out1[order], out2[order] = top1, other
    return out1, out2


def _rank_desc(group_keys, values):
    return (pd.Series(values).groupby(group_keys).rank(ascending=False, method="first")
            .to_numpy(np.float32))


def build_stage2(s_row, t_row, p1, carry: dict, tgt: pd.DataFrame, n_jobs: int = 4) -> np.ndarray:
    """Returns the stage-2 feature matrix aligned with the pair arrays."""
    n = len(p1)
    p1 = np.asarray(p1, np.float32)
    F = {c: np.asarray(carry[c], np.float32) for c in CARRY}
    F["p1"] = p1
    pc = np.clip(p1, 1e-6, 1 - 1e-6)
    F["p1_logit"] = np.log(pc / (1 - pc)).astype(np.float32)

    # ---- S1 context on p1
    mx, other_s = top2_by_group(s_row, p1)
    F["p1_rank_s"] = _rank_desc(s_row, p1)
    F["p1_gap_s"] = (mx - p1).astype(np.float32)
    F["p1_second_s"] = other_s
    F["p1_mass_other_s"] = (pd.Series(p1).groupby(s_row).transform("sum").to_numpy() - p1).astype(np.float32)
    F["p1_n50_s"] = pd.Series(p1 >= 0.5).groupby(s_row).transform("sum").to_numpy(np.float32)

    # ---- target competition on p1 (one-owner)
    _, F["p1_other_max_t"] = top2_by_group(t_row, p1)
    F["p1_gap_t"] = (p1 - F["p1_other_max_t"]).astype(np.float32)
    F["p1_n10_t"] = pd.Series(p1 >= 0.1).groupby(t_row).transform("sum").to_numpy(np.float32)

    # ---- sibling similarity
    for c in ("sib_name_max", "sib_addr_max", "sib_num_max", "sib_score", "sib_p1_best"):
        F[c] = np.zeros(n, np.float32)
    top = pd.DataFrame({"s": s_row, "t": t_row, "p": p1, "rk": F["p1_rank_s"]})
    top = top[(top["rk"] <= SIB_TOP) & (top["p"] >= 0.3)][["s", "t", "p"]]
    cand = pd.DataFrame({"row": np.arange(n), "s": s_row, "t": t_row})[p1 >= SIB_MIN_P1]
    pairs = cand.merge(top, on="s", suffixes=("", "_sib"))
    pairs = pairs[pairs["t"] != pairs["t_sib"]]
    if len(pairs):
        v = worker_view(tgt, ["n_skel", "a_atext", "a_nums"])
        _G.update(ta=pairs["t"].to_numpy(), tb=pairs["t_sib"].to_numpy(),
                  skel=v["n_skel"], atext=v["a_atext"], nums=v["a_nums"])
        m = len(pairs)
        sims = np.empty((m, 3), np.float32)
        for lo, arr in run_pool(_sib_worker, [(lo, min(lo + 50000, m)) for lo in range(0, m, 50000)], n_jobs):
            sims[lo:lo + len(arr)] = arr
        _G.clear()
        pairs = pairs.assign(sn=sims[:, 0], sa=sims[:, 1], su=sims[:, 2])
        pairs["score"] = pairs["p"] * (0.5 * pairs["sn"] + 0.5 * pairs["sa"]) / 100.0
        g = pairs.groupby("row")
        agg = g.agg(sn=("sn", "max"), sa=("sa", "max"), su=("su", "max"), sc=("score", "max"))
        best = pairs.loc[g["score"].idxmax(), ["row", "p"]].set_index("row")["p"]
        rows = agg.index.to_numpy()
        F["sib_name_max"][rows] = agg["sn"].to_numpy()
        F["sib_addr_max"][rows] = agg["sa"].to_numpy()
        F["sib_num_max"][rows] = agg["su"].to_numpy()
        F["sib_score"][rows] = agg["sc"].to_numpy()
        F["sib_p1_best"][best.index.to_numpy()] = best.to_numpy()
    return np.column_stack([F[c] for c in STAGE2_FEATURES]).astype(np.float32)

In [ ]:
%%writefile entity_matching/metrics.py
"""F-beta evaluation (default beta = 0.5: precision weighted 2x recall)."""
from __future__ import annotations

import numpy as np
import pandas as pd


def fbeta(p: float, r: float, beta: float = 0.5) -> float:
    b2 = beta * beta
    return (1 + b2) * p * r / (b2 * p + r) if (p + r) > 0 else 0.0


def evaluate(pred: pd.DataFrame, truth: pd.DataFrame, s1_ids, beta: float = 0.5) -> dict:
    """pred / truth: DataFrames[source1_entity_id, matched_id].
    Only S1 ids in `s1_ids` are scored.  Returns micro (pair-level) and macro
    (per-S1 averaged, empty-vs-empty = 1) scores."""
    s1_ids = pd.Index(pd.unique(np.asarray(s1_ids)))
    pred = pred[pred["source1_entity_id"].isin(s1_ids)]
    truth = truth[truth["source1_entity_id"].isin(s1_ids)]
    pk = set(zip(pred["source1_entity_id"], pred["matched_id"]))
    tk = set(zip(truth["source1_entity_id"], truth["matched_id"]))
    tp = len(pk & tk)
    p = tp / len(pk) if pk else 1.0
    r = tp / len(tk) if tk else 1.0

    # per-S1
    tp_s = pd.Series([a for a, _ in pk & tk], dtype=object).value_counts()
    np_s = pred["source1_entity_id"].value_counts()
    nt_s = truth["source1_entity_id"].value_counts()
    tp_v = tp_s.reindex(s1_ids).fillna(0).to_numpy()
    np_v = np_s.reindex(s1_ids).fillna(0).to_numpy()
    nt_v = nt_s.reindex(s1_ids).fillna(0).to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        ps = np.where(np_v > 0, tp_v / np_v, np.where(nt_v > 0, 0.0, 1.0))
        rs = np.where(nt_v > 0, tp_v / nt_v, np.where(np_v > 0, 0.0, 1.0))
        b2 = beta * beta
        fs = np.where((ps + rs) > 0, (1 + b2) * ps * rs / (b2 * ps + rs), 0.0)
    both_empty = (np_v == 0) & (nt_v == 0)
    fs = np.where(both_empty, 1.0, fs)
    return {
        "micro_precision": p, "micro_recall": r, "micro_fbeta": fbeta(p, r, beta),
        "macro_precision": float(ps.mean()), "macro_recall": float(rs.mean()),
        "macro_fbeta": float(fs.mean()), "tp": tp, "n_pred": len(pk), "n_true": len(tk),
    }

In [ ]:
%%writefile entity_matching/synth.py
"""
Synthetic data in the exact challenge layout, reproducing the noise found
in the EDA notebook.  Used for smoke-testing the pipeline end-to-end (the
real Kaggle data is not bundled).  Scores on synthetic data are NOT an
estimate of leaderboard scores.
"""
from __future__ import annotations

import random
from pathlib import Path

import pandas as pd

US_W1 = ["Summit", "Harbor", "Meridian", "Pioneer", "Cascade", "Sterling", "Keystone", "Apex",
         "Coastal", "Highland", "Silver", "Golden", "Bright", "Heritage", "Metro", "Blue",
         "Liberty", "Frontier", "Evergreen", "Oakwood", "Riverside", "Granite", "Beacon", "Vista"]
US_W2 = ["Pediatric", "Dental", "Eye", "Chiropractic", "Family", "Urology", "Vision",
         "Orthopedic", "Seafood", "Auto Body", "Printing", "Realty", "Capital", "Logistics",
         "Consulting", "Cargo", "Marketing", "Software", "Roofing", "Insurance"]
US_W3 = ["Group", "Associates", "Partners", "Center", "Care", "Clinic", "Services", "Holdings",
         "Solutions", "Works"]
US_LEGAL = ["LLC", "Inc", "Corp", "Co", "LP", "PC", "PLLC", "Ltd", ""]
SURNAMES = ["Smith", "Jones", "Garcia", "Kern", "Delavega", "Hawkins", "Brunson", "Murphy",
            "Townsend", "Graham", "Figueroa", "Prater", "Younker", "Wellman", "Holloway"]
US_STREETS = ["Westchester", "Ellis", "Montebello", "Cameron", "Elm", "Stardust", "Pierpont",
              "Mulberry", "Forest", "Moore", "Greyrock", "Meadow", "Allison", "Bell", "Oak",
              "Maple", "Cedar", "Lincoln", "Washington", "Park"]
US_TYPES = [("Street", "St"), ("Road", "Rd"), ("Drive", "Dr"), ("Avenue", "Ave"),
            ("Lane", "Ln"), ("Court", "Ct"), ("Trail", "Trl"), ("Boulevard", "Blvd")]
US_CITIES = [("High Point", "NC", "North Carolina"), ("Tahlequah", "OK", "Oklahoma"),
             ("Phoenix", "AZ", "Arizona"), ("Dundalk", "MD", "Maryland"),
             ("Cleveland", "OH", "Ohio"), ("Houston", "TX", "Texas"),
             ("Brooklyn", "NY", "New York"), ("Springfield", "IL", "Illinois"),
             ("Raleigh", "NC", "North Carolina"), ("Eugene", "OR", "Oregon")]

IN_W1 = ["Shakti", "Lakshmi", "Supreme", "Sky", "Green", "Arihant", "Jain", "Global", "Royal",
         "Modern", "Sun", "Krishna", "Balaji", "Great", "Prime", "Apex", "Shivam", "Om"]
IN_W2 = ["Technology", "Builders", "Consultants", "Foods", "Infra", "Marketing", "Ventures",
         "Enterprises", "Exports", "Solutions", "Estate", "Agro", "Finance"]
IN_LEGAL = ["Private Limited", "Pvt Ltd", "Limited", "LLP", "Pvt. Ltd.", ""]
DEVANAGARI = {
    "Shakti": "शक्ति", "Lakshmi": "लक्ष्मी", "Supreme": "सुप्रीम", "Sky": "स्काई",
    "Green": "ग्रीन", "Arihant": "अरिहंत", "Jain": "जैन", "Global": "ग्लोबल", "Royal": "रॉयल",
    "Modern": "मॉडर्न", "Sun": "सन", "Krishna": "कृष्णा", "Balaji": "बालाजी", "Great": "ग्रेट",
    "Prime": "प्राइम", "Apex": "एपेक्स", "Shivam": "शिवम", "Om": "ओम",
    "Technology": "टेक्नोलॉजी", "Builders": "बिल्डर्स", "Consultants": "कंसल्टेंट्स",
    "Foods": "फूड्स", "Infra": "इंफ्रा", "Marketing": "मार्केटिंग", "Ventures": "वेंचर्स",
    "Enterprises": "एंटरप्राइजेज", "Exports": "एक्सपोर्ट्स", "Solutions": "सॉल्यूशंस",
    "Estate": "एस्टेट", "Agro": "एग्रो", "Finance": "फाइनेंस",
    "Private": "प्राइवेट", "Limited": "लिमिटेड", "Pvt": "प्रा.", "Ltd": "लि.", "LLP": "एलएलपी",
}
IN_AREAS = ["Gulmohar Colony", "Karve Nagar", "Anand Nagar", "Jayanagar", "Kamala Nagar",
            "Andheri East", "Sector 42", "Rajaji Nagar", "Salt Lake", "Banjara Hills"]
IN_CITIES = [("Mumbai", "MH", "Maharashtra", "महाराष्ट्र"), ("Pune", "MH", "Maharashtra", "महाराष्ट्र"),
             ("Bangalore", "KA", "Karnataka", "ಕರ್ನಾಟಕ"), ("Chennai", "TN", "Tamil Nadu", "தமிழ்நாடு"),
             ("Hyderabad", "TG", "Telangana", "తెలంగాణ"), ("Kolkata", "WB", "West Bengal", "পশ্চিমবঙ্গ"),
             ("Ahmedabad", "GJ", "Gujarat", "ગુજરાત"), ("New Delhi", "DL", "Delhi", "दिल्ली")]
IN_LABELS = ["No.", "H.No", "Plot No", "Door No", "Flat No", "Shop No", "Office No"]

FR_W1 = ["Amicale", "Club", "Ecole", "Maison", "Comite", "Union", "Groupe", "Centre",
         "Pharmacie", "Institut", "Association", "Foyer"]
FR_W2 = ["Sportive", "Culturelle", "des Parents", "Bouliste", "du Lac", "Saint-Michel",
         "Laique", "Rotary", "des Amis", "Primaire"]
FR_LEGAL = ["SARL", "SAS", "SASU", "EURL", "SA", "SCI", ""]
FR_STREETS = ["Pierre Dignac", "Parmentier", "Lachassaigne", "de Cassel", "du Chaufour",
              "Jules Lefebvre", "Roger Salengro", "du President Wilson", "de la Renaudiere"]
FR_TYPES = [("Rue", "R"), ("Avenue", "Av"), ("Boulevard", "BD"), ("Allee", "All"),
            ("Impasse", "Imp"), ("Chemin", "Ch")]
FR_CITIES = [("Bordeaux", "Nouvelle-Aquitaine"), ("Nantes", "Pays de la Loire"),
             ("Lille", "Hauts-de-France"), ("Dunkerque", "Hauts-de-France"),
             ("Calais", "Hauts-de-France"), ("Pessac", "Nouvelle-Aquitaine")]

MATCH_COUNT_DIST = [(0, 5.6), (1, 5.4), (2, 17.0), (3, 24.1), (4, 21.9), (5, 14.6),
                    (6, 7.5), (7, 2.9), (8, 0.85), (9, 0.2), (10, 0.03)]
ACCENTS = {"a": "á", "e": "é", "i": "í", "o": "ó", "u": "ú", "c": "ç"}
OCR = {"o": "0", "l": "1", "s": "5", "i": "l", "e": "3"}


class Gen:
    def __init__(self, seed):
        self.r = random.Random(seed)
        self.used_ids = set()

    def uid(self, prefix):
        while True:
            i = self.r.randint(10_000, 999_999_999)
            if i not in self.used_ids:
                self.used_ids.add(i)
                return f"{prefix}-{i}"

    # ---------------- clean entities ----------------
    def entity(self, country):
        r = self.r
        if country == "US":
            if r.random() < 0.35:  # generic -> hard negatives (cell 20)
                words = [r.choice(US_W2[:8]), r.choice(["Group", "Associates", "Center", "Care"])]
            elif r.random() < 0.2:
                a, b = r.sample(SURNAMES, 2)
                words = [f"{a},", b, "&", r.choice(SURNAMES)]
            else:
                words = [r.choice(US_W1), r.choice(US_W2)] + ([r.choice(US_W3)] if r.random() < .6 else [])
            legal = r.choice(US_LEGAL)
            city, code, state = r.choice(US_CITIES)
            num = str(r.randint(1, 29999))
            st, _ = r.choice(US_TYPES)
            addr = {"num": num, "street": r.choice(US_STREETS), "type": st, "city": city,
                    "code": code, "state": state,
                    "unit": (f"Unit {r.randint(1, 900)}" if r.random() < 0.2 else "")}
        elif country == "India":
            words = [r.choice(IN_W1), r.choice(IN_W2)]
            if r.random() < 0.3:
                words.insert(1, r.choice(IN_W1))
            legal = r.choice(IN_LEGAL)
            city, code, state, native = r.choice(IN_CITIES)
            n = str(r.randint(1, 999))
            num = n + (f"/{r.randint(1, 99)}" if r.random() < 0.5 else "")
            addr = {"label": r.choice(IN_LABELS), "num": num, "area": r.choice(IN_AREAS),
                    "city": city, "code": code, "state": state, "native": native,
                    "floor": (r.choice(["1st Floor", "2nd Floor", "Ground Floor"]) if r.random() < .3 else "")}
        else:
            words = [r.choice(FR_W1), r.choice(FR_W2)]
            legal = r.choice(FR_LEGAL)
            city, region = r.choice(FR_CITIES)
            st, _ = r.choice(FR_TYPES)
            addr = {"num": str(r.randint(1, 400)), "bis": r.random() < 0.1, "type": st,
                    "street": r.choice(FR_STREETS), "city": city, "region": region}
        return {"country": country, "words": words, "legal": legal, "addr": addr}

    def render_name(self, e):
        return " ".join(e["words"] + ([e["legal"]] if e["legal"] else [])).replace(" ,", ",")

    def render_addr(self, e, noisy=False):
        r, a, c = self.r, e["addr"], e["country"]
        if c == "US":
            typ = a["type"]
            if noisy and r.random() < 0.5:
                typ = dict(US_TYPES)[typ]
            num = a["num"]
            if noisy and r.random() < 0.1:
                num = "0" + num
            if noisy and r.random() < 0.05:
                num = "#" + num
            state = a["code"] if (not noisy or r.random() < 0.5) else a["state"]
            comps = [f"{num} {a['street']} {typ}"] + ([a["unit"]] if a["unit"] else []) + [a["city"], state]
        elif c == "India":
            lab = a["label"] if not noisy else r.choice(IN_LABELS + [a["label"]] * 2)
            state = a["state"]
            if noisy:
                state = r.choice([a["state"], a["code"], a["native"]])
            comps = [f"{lab} {a['num']}"] + ([a["floor"]] if a["floor"] else []) + [a["area"], a["city"], state]
        else:
            typ = a["type"]
            if noisy and r.random() < 0.4:
                typ = dict(FR_TYPES)[typ]
            num = a["num"] + (" bis" if a["bis"] else "")
            comps = [f"{num} {typ} {a['street']}", a["city"], a["region"]]
        if noisy:
            if r.random() < 0.35:
                r.shuffle(comps)
            if r.random() < 0.08:
                comps.insert(r.randint(0, len(comps)), r.choice(["NULL", "null", "<NULL>"]))
            if r.random() < 0.1 and len(comps) > 2:
                comps.pop(r.randrange(len(comps)))
            s = ", ".join(comps)
            if r.random() < 0.3:
                s = s.upper()
            if r.random() < 0.15:
                s = self.typo(s)
            if r.random() < 0.035:
                s = ""
            return s
        return ", ".join(comps)

    # ---------------- noise ops ----------------
    def typo(self, s):
        r = self.r
        if len(s) < 4:
            return s
        i = r.randrange(1, len(s) - 1)
        op = r.random()
        if op < 0.3:
            return s[:i] + s[i + 1:]
        if op < 0.6:
            return s[:i] + r.choice("abcdefghilmnorstu") + s[i:]
        if op < 0.8:
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
        return s[:i] + r.choice("aeiou") + s[i + 1:]

    def noisy_name(self, e):
        r = self.r
        words = list(e["words"])
        legal = e["legal"]
        c = e["country"]
        if r.random() < 0.02:  # name replaced entirely (cell 17: "Kelopyrahalo")
            return r.choice(["Kelopyrahalo", "Noviveo", "Ectoecto", "Viovera", "Jaxzeta"])
        if c == "India" and r.random() < 0.2 and all(w in DEVANAGARI for w in words):
            parts = [DEVANAGARI[w] for w in words]
            for lw in legal.replace(".", "").split():
                parts.append(DEVANAGARI.get(lw, lw))
            if r.random() < 0.3:  # mixed script
                parts[0] = words[0]
            return " ".join(parts)
        ops = r.sample(range(14), k=r.choice([1, 1, 2, 2, 3]))
        for op in ops:
            if op == 0 and legal:
                legal = r.choice({"US": US_LEGAL, "India": IN_LEGAL, "France": FR_LEGAL}[c])
            elif op == 1 and legal:
                legal = r.choice([f"({legal})", f"[{legal}]", f"([{legal}])", legal.upper()])
            elif op == 2 and legal:
                words = [legal] + words
                legal = ""
            elif op == 3:
                i = r.randrange(len(words))
                words[i] = self.typo(words[i])
            elif op == 4:
                i = r.randrange(len(words))
                words[i] = "".join(OCR.get(ch, ch) if r.random() < 0.3 else ch for ch in words[i])
            elif op == 5:
                i = r.randrange(len(words))
                words[i] = "".join(ACCENTS.get(ch, ch) if r.random() < 0.2 else ch for ch in words[i])
            elif op == 6:
                i = r.randrange(len(words))
                words.insert(i, words[i])
            elif op == 7 and len(words) > 1:
                r.shuffle(words)
            elif op == 8:
                name = "".join(w.strip(",&") for w in words).lower()
                return name + r.choice([".com", ".Com", ".c0m", ".net"])
            elif op == 9:
                alias = r.choice(["Vantagexylo D.B.A.", "Korzephdova formerly:", "T/A"])
                return f"{alias} {' '.join(words)} {legal}".strip()
            elif op == 10:
                words.append(r.choice(["Partners", "Center", "Group", "Co", "Enterprises"]))
            elif op == 11:
                return f"{' '.join(words)} {legal} (ID: {r.randint(10000, 99999)})".strip()
            elif op == 12:
                words = [r.choice(["***", "--", ">>", "M/s"])] + words
            elif op == 13:
                legal = ""
        s = " ".join(words + ([legal] if legal else []))
        rr = r.random()
        if rr < 0.2:
            s = s.upper()
        elif rr < 0.3:
            s = s.lower()
        elif rr < 0.35:
            s = s.replace(" ", "  ", 1)
        return s

    # ---------------- dataset ----------------
    def build(self, n_s1, countries):
        s1, s2, s3, gt = [], [], [], []
        counts, weights = zip(*MATCH_COUNT_DIST)
        for _ in range(n_s1):
            c = self.r.choices(countries[0], weights=countries[1])[0]
            e = self.entity(c)
            sid = self.uid("S1")
            s1.append((sid, self.render_name(e), self.render_addr(e), c))
            k = self.r.choices(counts, weights=weights)[0]
            ids = []
            for _ in range(k):
                src = self.r.choice(["S2", "S3"])
                tid = self.uid(src)
                row = (tid, self.noisy_name(e), self.render_addr(e, noisy=True), c)
                (s2 if src == "S2" else s3).append(row)
                ids.append(tid)
            gt.append((sid, ",".join(ids)))
        # distractor targets: entities absent from S1 (~30 % of targets)
        n_dis = int(0.3 * (len(s2) + len(s3)))
        for _ in range(n_dis):
            c = self.r.choices(countries[0], weights=countries[1])[0]
            e = self.entity(c)
            src = self.r.choice(["S2", "S3"])
            row = (self.uid(src), self.noisy_name(e), self.render_addr(e, noisy=True), c)
            (s2 if src == "S2" else s3).append(row)
        cols = ["entity_id", "business_name", "business_address", "country"]
        self.r.shuffle(s2)
        self.r.shuffle(s3)
        return (pd.DataFrame(s1, columns=cols), pd.DataFrame(s2, columns=cols),
                pd.DataFrame(s3, columns=cols),
                pd.DataFrame(gt, columns=["source1_entity_id", "matched_entity_ids"]))


def write_synthetic(out_dir, n_train=6000, n_test=3000, seed=0):
    out = Path(out_dir)
    g = Gen(seed)
    for split, n, countries in [("train", n_train, (["US", "India"], [60, 40])),
                                ("test", n_test, (["US", "India", "France"], [38, 47, 15]))]:
        d = out / split
        d.mkdir(parents=True, exist_ok=True)
        s1, s2, s3, gt = g.build(n, countries)
        for name, df in [("source1", s1), ("source2", s2), ("source3", s3), ("ground_truth", gt)]:
            df.to_csv(d / f"{split}_{name}.tsv", sep="\t", index=False)
        print(f"{split}: S1={len(s1):,} S2={len(s2):,} S3={len(s3):,} links={gt.matched_entity_ids.str.count('S').sum():,}")

In [ ]:
%%writefile entity_matching/charnn.py
"""
Deep ensemble member: character-level cross-attention matcher (ESIM-style)
fused with the tabular features (wide & deep).

Each record -> 96 bytes: normalised name (32) + normalised address (64); both
are ASCII after transliteration.  A shared char-CNN encodes each side, soft
cross-attention aligns every character position of one side with the other
side (Chen et al. 2017, ESIM; Mudgal et al. 2018, DeepMatcher), the aligned
comparisons [h - a, h * a] are mean/max pooled, and an MLP fuses them with
the standardised tabular features.  It learns typo / transliteration /
reordering tolerance end-to-end, which gives the gradient-boosting members
a genuinely different error profile to average with.

Only used in phase C (after every fork), trained fold-wise with the same
GroupKFold split so its out-of-fold predictions are comparable, and chosen
into the blend only if OOF macro F0.5 improves.  A wall-clock budget cuts
epochs or skips the member instead of risking the deadline.
"""
from __future__ import annotations

import math
import time

import numpy as np
import pandas as pd

from .utils import log

NAME_LEN, ADDR_LEN = 32, 64
SEQ = NAME_LEN + ADDR_LEN


def _encode_unique(values, width):
    """Unique strings -> (n, width) uint8 matrix (ASCII, zero padded)."""
    buf = b"".join(str(v).encode("ascii", "replace")[:width].ljust(width, b"\0") for v in values)
    return np.frombuffer(buf, dtype=np.uint8).reshape(len(values), width).copy() if len(values) \
        else np.zeros((0, width), np.uint8)


class TextIndex:
    """Per-record byte text without materialising 10M strings: unique-value
    matrices + per-record codes (from the categorical columns)."""

    def __init__(self, df: pd.DataFrame, name_col="n_core", addr_col="a_atext"):
        parts = []
        for col, width in ((name_col, NAME_LEN), (addr_col, ADDR_LEN)):
            s = df[col]
            if not isinstance(s.dtype, pd.CategoricalDtype):
                s = s.astype("category")
            parts.append((_encode_unique(list(s.cat.categories), width),
                          s.cat.codes.to_numpy().astype(np.int32)))
        (self.nm, self.nc), (self.am, self.ac) = parts

    def to_arrays(self, prefix):
        return {prefix + "nm": self.nm, prefix + "nc": self.nc, prefix + "am": self.am, prefix + "ac": self.ac}

    @classmethod
    def from_arrays(cls, z, prefix):
        obj = cls.__new__(cls)
        obj.nm, obj.nc, obj.am, obj.ac = (z[prefix + k] for k in ("nm", "nc", "am", "ac"))
        return obj

    def rows(self, idx):
        idx = np.asarray(idx)
        n = self.nm[np.maximum(self.nc[idx], 0)]
        n[self.nc[idx] < 0] = 0
        a = self.am[np.maximum(self.ac[idx], 0)]
        a[self.ac[idx] < 0] = 0
        return np.concatenate([n, a], axis=1)


def _build_model(n_tab):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class CharMatcher(nn.Module):
        def __init__(self, d=96, e=48):
            super().__init__()
            self.d = d
            self.emb = nn.Embedding(256, e, padding_idx=0)
            self.pos = nn.Parameter(torch.zeros(1, SEQ, e))
            self.c1 = nn.Conv1d(e, d, 5, padding=2)
            self.c2 = nn.Conv1d(d, d, 3, padding=1)
            self.c3 = nn.Conv1d(d, d, 3, padding=2, dilation=2)
            self.tab = nn.Sequential(nn.Linear(n_tab, 192), nn.GELU(), nn.Linear(192, 96), nn.GELU())
            self.head = nn.Sequential(nn.Linear(8 * d + 96, 256), nn.GELU(), nn.Dropout(0.1),
                                      nn.Linear(256, 64), nn.GELU(), nn.Linear(64, 1))

        def enc(self, x):
            m = (x > 0)
            h = self.emb(x.long()) + self.pos
            h = F.gelu(self.c1(h.transpose(1, 2)))
            h = F.gelu(self.c2(h)) + h
            h = F.gelu(self.c3(h)) + h
            return h.transpose(1, 2), m

        @staticmethod
        def _pool(f, m):
            mf = m.unsqueeze(-1).to(f.dtype)
            mean = (f * mf).sum(1) / mf.sum(1).clamp_min(1.0)
            mx = f.masked_fill(~m.unsqueeze(-1), -1e4).max(1).values
            return torch.cat([mean, mx], -1)

        def forward(self, a, b, t):
            ha, ma = self.enc(a)
            hb, mb = self.enc(b)
            att = torch.bmm(ha, hb.transpose(1, 2)) / math.sqrt(self.d)
            wa = att.masked_fill(~mb.unsqueeze(1), -1e4).softmax(-1)
            wb = att.transpose(1, 2).masked_fill(~ma.unsqueeze(1), -1e4).softmax(-1)
            al_a = torch.bmm(wa, hb)
            al_b = torch.bmm(wb, ha)
            fa = torch.cat([ha - al_a, ha * al_a], -1)
            fb = torch.cat([hb - al_b, hb * al_b], -1)
            z = torch.cat([self._pool(fa, ma), self._pool(fb, mb), self.tab(t)], -1)
            return self.head(z).squeeze(-1)

    return CharMatcher()


def nn_oof(text_tr: TextIndex, s_tr, t_tr_text: TextIndex, t_tr, X, y, folds, ok,
           text_te: TextIndex, s_te, t_te_text: TextIndex, t_te, Xt,
           gpu: bool, n_jobs: int, seed: int = 42, budget_s: float = 1200.0,
           max_epochs: int = 4, batch: int = 512):
    """Fold-wise training -> (oof probs, test probs = mean of fold models)."""
    import torch
    import torch.nn.functional as F

    t_start = time.time()
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.set_num_threads(max(1, n_jobs))
    dev = torch.device("cuda" if gpu and torch.cuda.is_available() else "cpu")
    amp = dev.type == "cuda"

    mu = np.nanmean(X, 0)
    sd = np.nanstd(X, 0) + 1e-6

    def tab(Z):
        return np.clip(np.nan_to_num((Z - mu) / sd), -5, 5).astype(np.float32)

    A = torch.from_numpy(text_tr.rows(s_tr)).to(dev)
    B = torch.from_numpy(t_tr_text.rows(t_tr)).to(dev)
    T = torch.from_numpy(tab(X)).to(dev)
    Y = torch.from_numpy(y.astype(np.float32)).to(dev)

    def predict(model, a, b, t):
        model.eval()
        out = []
        with torch.no_grad(), torch.autocast(dev.type, enabled=amp):
            for i in range(0, len(t), 4096):
                out.append(torch.sigmoid(model(a[i:i + 4096], b[i:i + 4096], t[i:i + 4096]).float()))
        return torch.cat(out).cpu().numpy() if out else np.zeros(0, np.float32)

    oof = np.zeros(len(y), np.float32)
    models = []
    n_folds = len(folds)
    per_fold_budget = budget_s / (n_folds + 1)  # keep ~1 share for test inference
    for k, (tr, va) in enumerate(folds):
        tr = tr[ok[tr]]
        model = _build_model(X.shape[1]).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
        steps_per_epoch = max(1, math.ceil(len(tr) / batch))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps_per_epoch * max_epochs,
                                                    pct_start=0.15)
        scaler = torch.amp.GradScaler(dev.type, enabled=amp)
        tr_t = torch.from_numpy(tr).to(dev)
        va_t = torch.from_numpy(va).to(dev)
        best, best_state, f0 = np.inf, None, time.time()
        for ep in range(max_epochs):
            model.train()
            perm = tr_t[torch.randperm(len(tr_t), device=dev)]
            for i in range(0, len(perm), batch):
                bi = perm[i:i + batch]
                with torch.autocast(dev.type, enabled=amp):
                    loss = F.binary_cross_entropy_with_logits(model(A[bi], B[bi], T[bi]).float(), Y[bi])
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
            pv = predict(model, A[va_t], B[va_t], T[va_t])
            yv = y[va]
            ll = float(-np.mean(yv * np.log(np.clip(pv, 1e-6, 1)) + (1 - yv) * np.log(np.clip(1 - pv, 1e-6, 1))))
            if ll < best:
                best, best_state = ll, {kk: v.detach().clone() for kk, v in model.state_dict().items()}
                oof[va] = pv
            el = time.time() - f0
            log(f"    [charnn-{dev.type}] fold {k} epoch {ep + 1}: val logloss {ll:.5f} ({el:.0f}s)")
            if el / (ep + 1) * (ep + 2) > per_fold_budget:  # next epoch would exceed the budget
                break
        model.load_state_dict(best_state)
        models.append(model)
        if time.time() - t_start > budget_s * (k + 2) / (n_folds + 1) and k < n_folds - 1:
            raise TimeoutError("charnn over budget - member skipped")
    del A, B, T, Y
    # test: mean of the fold models (no refit -> bounded time), streamed in
    # chunks so millions of test pairs never sit in memory as tensors
    pt = np.zeros(len(s_te), np.float32)
    for i in range(0, len(s_te), 262144):
        sl = slice(i, i + 262144)
        At = torch.from_numpy(text_te.rows(s_te[sl])).to(dev)
        Bt = torch.from_numpy(t_te_text.rows(t_te[sl])).to(dev)
        Tt = torch.from_numpy(tab(Xt[sl])).to(dev)
        pt[sl] = np.mean([predict(m, At, Bt, Tt) for m in models], axis=0)
        del At, Bt, Tt
    del models
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    log(f"    [charnn] done in {time.time() - t_start:.0f}s")
    return oof, pt

In [ ]:
%%writefile entity_matching/final.py
"""
FINAL competition pipeline (v3): learned two-stage matcher + GPU ensemble.

Phase A  train  (CPU, fork pools)  normalise -> block -> features ->
                                    stage-1 LightGBM (GroupKFold OOF) ->
                                    learned pruning rule chosen on OOF
Phase B  test   (CPU, fork pools)  normalise -> block -> features ->
                                    stage-1 score -> prune -> candidate_pairs.tsv
Phase C  models (GPU allowed)      LightGBM + XGBoost + CatBoost on the pruned
                                    pairs (OOF), blend chosen on OOF, isotonic
                                    calibration, macro-F0.5 decision rule
                                    -> matching_results.tsv

Deadlock safety: every fork happens in phases A/B.  CUDA (XGBoost/CatBoost
GPU) is first touched in phase C, after which nothing forks again.  Forking
a process that already holds a CUDA context is the classic hang.

Metric (problem statement): F0.5 per Source-1 entity, macro-averaged over
ALL S1 entities; a singleton predicted empty scores 1.0.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from .blocking import BlockConfig, generate_candidates
from .charnn import TextIndex, nn_oof
from .data import gt_pairs, load_split, normalize_frame
from .decision import Calibrator, expected_f_select, threshold_select
from .features import ALL_FEATURES, add_s1_context, build_idf, compute_pair_features
from .utils import auto_data_dir, clear_strcol_cache, free, is_kaggle, log, n_workers

# blocking-based target-competition features need blocking over ALL S1
# records; training blocks only the sampled S1s, so they are left out
# (one-owner is still enforced by the decision rule)
FEATS = [f for f in ALL_FEATURES if f not in ("ctx_ncand_t", "ctx_rank_t", "ctx_gap_t")]
DOUBT_COLS = ["n_tset_skel", "n_idf_contain", "a_tset", "num_shared", "n_alt_best",
              "n_exact_skel", "a_missing_t", "num_jacc", "num_s"]


@dataclass
class FinalConfig:
    data_dir: str | None = None
    out_dir: str | None = None
    train_s1_sample: int = 250_000     # S1 records used for training (0 = all)
    folds: int = 3
    beta: float = 0.5
    n_jobs: int | None = None
    prune_keep: float = 0.998          # pruning keeps >= 99.8 % of reachable true links
    gpu: str = "auto"                  # auto | on | off
    use_xgboost: bool = True
    use_catboost: bool = True
    use_nn: bool = True                # char cross-attention member (GPU; CPU only if nn_on_cpu)
    nn_on_cpu: bool = False
    nn_budget_s: float = 1200.0
    per_country_rules: bool = True
    batch_s1: int = 100_000
    seed: int = 42
    team_name: str = "team"
    make_zip: bool = True


# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------
def gpu_available() -> bool:
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, timeout=20)
        return r.returncode == 0 and b"GPU" in r.stdout
    except Exception:
        return False


def rank_in_s(s_row, v):
    order = np.lexsort((-v, s_row))
    ss = s_row[order]
    first = np.r_[True, ss[1:] != ss[:-1]] if len(ss) else np.zeros(0, bool)
    start = np.maximum.accumulate(np.where(first, np.arange(len(ss)), 0))
    r = np.empty(len(v), np.int64)
    r[order] = np.arange(len(ss)) - start
    return r


def macro_micro(s_row, y, sel, n_true, beta=0.5, univ=None):
    """F-beta per S1 averaged over ALL S1 in the universe (len(n_true)).
    n_true[s] counts every true link of S1 s, including ones blocking or
    pruning lost (they are false negatives)."""
    n = len(n_true)
    tp = np.bincount(s_row[sel], weights=y[sel].astype(float), minlength=n)
    npred = np.bincount(s_row[sel], minlength=n).astype(float)
    b2 = beta * beta
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(npred > 0, tp / npred, np.where(n_true > 0, 0.0, 1.0))
        r = np.where(n_true > 0, tp / n_true, np.where(npred > 0, 0.0, 1.0))
        f = np.where((p + r) > 0, (1 + b2) * p * r / (b2 * p + r), 0.0)
    f = np.where((npred == 0) & (n_true == 0), 1.0, f)
    if univ is not None:  # average over a subset of S1 (e.g. one country)
        f, p, r = f[univ], p[univ], r[univ]
        tp, npred, n_true = tp[univ], npred[univ], n_true[univ]
    P = tp.sum() / npred.sum() if npred.sum() else 1.0
    R = tp.sum() / n_true.sum() if n_true.sum() else 1.0
    micro = (1 + b2) * P * R / (b2 * P + R) if (P + R) else 0.0
    return {"macro_f05": float(f.mean()), "macro_p": float(p.mean()), "macro_r": float(r.mean()),
            "micro_f05": float(micro), "micro_p": float(P), "micro_r": float(R)}


def apply_rule(rule, s_row, t_row, q, beta):
    if rule["rule"] == "threshold":
        return threshold_select(s_row, t_row, q, rule["thr"], rule["alpha"])
    return expected_f_select(s_row, t_row, q, beta, rule["lam"], rule["empty_bias"], rule["min_q"])


def tune_rule(s_row, t_row, y, q, n_true, beta, lam_hat, univ=None):
    best = None
    grid = [{"rule": "threshold", "thr": float(t), "alpha": a}
            for t in np.round(np.arange(0.15, 0.96, 0.025), 3) for a in (0.0, 0.3, 0.6)]
    grid += [{"rule": "expected_f", "lam": lam, "empty_bias": eb, "min_q": mq}
             for lam in (0.0, lam_hat) for eb in (0.5, 0.75, 1.0, 1.5, 2.0, 3.0) for mq in (0.0, 0.05)]
    for rule in grid:
        m = macro_micro(s_row, y, apply_rule(rule, s_row, t_row, q, beta), n_true, beta, univ)
        if best is None or m["macro_f05"] > best[1]["macro_f05"]:
            best = (rule, m)
    return best


def featurize(s1, tgt, s_rows, bcfg, n_jobs, idf, batch_s1, scorer=None):
    """Blocking + features in S1 batches.
    scorer(Xpart, s_row_part) -> keep-mask and p1; used at test time to prune
    each batch right away so only the pruned pairs are kept in memory."""
    t0 = time.time()
    cands = generate_candidates(s1, tgt, bcfg, s1_rows=s_rows, log=log)
    n_block = len(cands)
    log(f"  blocking: {n_block:,} pairs ({n_block / max(len(s_rows), 1):.1f} per S1) "
        f"in {time.time() - t0:.0f}s")
    # keep only targets that appear in some candidate pair (and trim the
    # categorical vocabularies): the feature stage then holds a fraction of
    # the 10M-record frame
    used = np.unique(cands["t_row"].to_numpy())
    clear_strcol_cache()
    tgt = tgt.iloc[used].reset_index(drop=True)
    for c in tgt.columns:
        if isinstance(tgt[c].dtype, pd.CategoricalDtype) and c not in ("country", "source"):
            tgt[c] = tgt[c].cat.remove_unused_categories()
    cands["t_row"] = np.searchsorted(used, cands["t_row"].to_numpy()).astype(np.int32)
    del used
    free()
    log(f"  targets kept for features: {len(tgt):,}")
    cands = cands.sort_values("s_row", kind="stable").reset_index(drop=True)
    srow = cands["s_row"].to_numpy()
    edges = np.r_[np.searchsorted(srow, np.unique(srow)[::batch_s1]), len(srow)]
    out = {"s": [], "t": [], "X": [], "p1": [], "doubt": []}
    for b, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        part = compute_pair_features(cands.iloc[lo:hi], s1, tgt, idf, n_jobs=n_jobs)
        part = add_s1_context(part, tgt)
        X = part[FEATS].to_numpy(np.float32)
        s_p, t_p = part["s_row"].to_numpy(np.int32), part["t_row"].to_numpy(np.int32)
        if scorer is not None:
            keep, p1 = scorer(X, s_p)
            out["p1"].append(p1[keep])
            X, s_p, t_p = X[keep], s_p[keep], t_p[keep]
        else:
            out["doubt"].append(part[DOUBT_COLS].to_numpy(np.float32))
        out["s"].append(s_p)
        out["t"].append(t_p)
        out["X"].append(X)
        del part
        free()
        log(f"  features batch {b + 1}/{len(edges) - 1}: {hi - lo:,} pairs -> kept {len(s_p):,}")
    del cands
    res = {k: (np.concatenate(v) if v else None) for k, v in out.items()}
    if res["X"] is None:
        res = {"s": np.zeros(0, np.int32), "t": np.zeros(0, np.int32),
               "X": np.zeros((0, len(FEATS)), np.float32), "p1": np.zeros(0, np.float32),
               "doubt": np.zeros((0, len(DOUBT_COLS)), np.float32)}
    res["n_block"] = n_block
    res["tgt"] = tgt  # the trimmed target frame; t indices refer to it
    return res


# ---------------------------------------------------------------------------
# models
# ---------------------------------------------------------------------------
def lgb_oof(X, y, folds, ok, params, rounds, tag, n_jobs):
    import lightgbm as lgb
    full = lgb.Dataset(X, y, free_raw_data=False, params={"max_bin": 255}).construct()
    oof = np.zeros(len(y), np.float32)
    its = []
    for k, (tr, va) in enumerate(folds):
        tr = np.sort(tr[ok[tr]])
        bst = lgb.train(params, full.subset(tr), num_boost_round=rounds,
                        valid_sets=[full.subset(np.sort(va))],
                        callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va] = bst.predict(X[va], num_iteration=bst.best_iteration, num_threads=n_jobs)
        its.append(bst.best_iteration or rounds)
        log(f"    [{tag}] fold {k}: {its[-1]} iters")
        del bst
        free()
    final = lgb.train(params, full.subset(np.flatnonzero(ok)), num_boost_round=int(np.mean(its) * 1.1) + 1)
    del full
    free()
    return oof, final


def xgb_oof(X, y, folds, ok, Xtest, gpu, n_jobs, seed):
    import xgboost as xgb
    params = {"objective": "binary:logistic", "eval_metric": "logloss", "eta": 0.08,
              "max_depth": 8, "min_child_weight": 5, "subsample": 0.8,
              "colsample_bytree": 0.8, "lambda": 1.0, "tree_method": "hist",
              "max_bin": 256, "seed": seed, "nthread": n_jobs}
    params["device"] = "cuda" if gpu else "cpu"
    oof = np.zeros(len(y), np.float32)
    its = []
    for k, (tr, va) in enumerate(folds):
        tr = tr[ok[tr]]
        dtr = xgb.DMatrix(X[tr], label=y[tr])
        dva = xgb.DMatrix(X[va], label=y[va])
        bst = xgb.train(params, dtr, 3000, evals=[(dva, "va")], early_stopping_rounds=50,
                        verbose_eval=False)
        oof[va] = bst.predict(dva, iteration_range=(0, bst.best_iteration + 1))
        its.append(bst.best_iteration + 1)
        log(f"    [xgb{'-gpu' if gpu else ''}] fold {k}: {its[-1]} iters")
        del dtr, dva, bst
        free()
    dall = xgb.DMatrix(X[ok], label=y[ok])
    bst = xgb.train(params, dall, int(np.mean(its) * 1.1) + 1, verbose_eval=False)
    pred = bst.predict(xgb.DMatrix(Xtest))
    del dall, bst
    free()
    return oof, pred.astype(np.float32)


def cat_oof(X, y, folds, ok, Xtest, gpu, n_jobs, seed):
    from catboost import CatBoostClassifier
    kw = dict(loss_function="Logloss", learning_rate=0.1 if gpu else 0.15,
              depth=8 if gpu else 6, iterations=3000 if gpu else 500,
              random_seed=seed, verbose=0, border_count=128,
              od_type="Iter", od_wait=50, allow_writing_files=False)
    if gpu:
        kw.update(task_type="GPU", devices="0")
    else:
        kw.update(thread_count=n_jobs)
    oof = np.zeros(len(y), np.float32)
    its = []
    for k, (tr, va) in enumerate(folds):
        tr = tr[ok[tr]]
        m = CatBoostClassifier(**kw)
        m.fit(X[tr], y[tr], eval_set=(X[va], y[va]), use_best_model=True)
        oof[va] = m.predict_proba(X[va])[:, 1]
        its.append(max(m.get_best_iteration() or 0, 1) + 1)
        log(f"    [cat{'-gpu' if gpu else ''}] fold {k}: {its[-1]} iters")
        del m
        free()
    kw.pop("od_type"), kw.pop("od_wait")
    kw["iterations"] = int(np.mean(its) * 1.1) + 1
    m = CatBoostClassifier(**kw)
    m.fit(X[ok], y[ok])
    pred = m.predict_proba(Xtest)[:, 1]
    del m
    free()
    return oof, pred.astype(np.float32)


# ---------------------------------------------------------------------------
# outputs
# ---------------------------------------------------------------------------
def write_lists(path, s1_ids, s_idx, t_ids_arr, col):
    df = pd.DataFrame({"s": s1_ids[s_idx], "t": t_ids_arr})
    df = df.drop_duplicates()
    agg = df.groupby("s", sort=False)["t"].agg(",".join)
    out = pd.DataFrame({"source1_entity_id": s1_ids})
    out[col] = out["source1_entity_id"].map(agg).fillna("")
    out.to_csv(path, sep="\t", index=False)
    return out


def validate_outputs(match_path, cand_path, test_s1_ids, test_target_ids):
    """Same rules as utils/validate_submission.py (problem statement)."""
    issues = []
    s1_set = set(test_s1_ids)
    tgt_set = set(test_target_ids)
    lists = {}
    for path, col in [(match_path, "matched_entity_ids"), (cand_path, "candidate_entity_ids")]:
        df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
        if list(df.columns) != ["source1_entity_id", col]:
            issues.append(f"{path.name}: bad header {list(df.columns)}")
        if df["source1_entity_id"].duplicated().any():
            issues.append(f"{path.name}: duplicate source1_entity_id rows")
        if set(df["source1_entity_id"]) != s1_set:
            issues.append(f"{path.name}: S1 ids differ from test_source1 "
                          f"(missing {len(s1_set - set(df['source1_entity_id']))})")
        d = {}
        for sid, ids in zip(df["source1_entity_id"], df[col]):
            li = [x for x in ids.split(",") if x] if ids else []
            if len(li) != len(set(li)):
                issues.append(f"{path.name}: duplicate ids in list of {sid}")
                break
            bad = [x for x in li if x not in tgt_set or not (x.startswith("S2-") or x.startswith("S3-"))]
            if bad:
                issues.append(f"{path.name}: unknown / non S2-S3 ids for {sid}: {bad[:3]}")
                break
            d[sid] = set(li)
        lists[col] = d
    m, c = lists.get("matched_entity_ids", {}), lists.get("candidate_entity_ids", {})
    not_sub = sum(1 for k, v in m.items() if not v <= c.get(k, set()))
    if not_sub:
        issues.append(f"{not_sub} S1 rows have matches that are not candidates")
    return issues


# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------
def _setup(cfg: FinalConfig):
    n_jobs = n_workers(cfg.n_jobs)
    out_dir = Path(cfg.out_dir or ("/kaggle/working" if is_kaggle() else "final_out"))
    (out_dir / "output").mkdir(parents=True, exist_ok=True)
    work = out_dir / "_work"
    work.mkdir(parents=True, exist_ok=True)
    return n_jobs, out_dir, work


def phase_a(cfg: FinalConfig):
    import lightgbm as lgb  # noqa: F401
    from sklearn.model_selection import GroupKFold
    T0 = time.time()
    n_jobs, out_dir, work = _setup(cfg)
    data_dir = auto_data_dir(cfg.data_dir, "train")
    log(f"data: {data_dir} | out: {out_dir} | workers: {n_jobs}")
    bcfg = BlockConfig(n_jobs=n_jobs)
    report = {"config": asdict(cfg)}
    # ===================== PHASE A: train (CPU) =====================
    log("PHASE A - train data")
    s1, tgt, gt = load_split(data_dir, "train")
    truth = gt_pairs(gt)
    del gt
    n_s1_total = len(s1)
    rng = np.random.default_rng(cfg.seed)
    if cfg.train_s1_sample and cfg.train_s1_sample < len(s1):
        s1 = s1.iloc[np.sort(rng.choice(len(s1), cfg.train_s1_sample, replace=False))]
    s1 = s1.reset_index(drop=True)
    log(f"  S1 used: {len(s1):,}  targets: {len(tgt):,}")
    normalize_frame(s1, n_jobs)
    normalize_frame(tgt, n_jobs)
    idf = build_idf(tgt)
    # reverse search (target -> its top-R S1) is only meaningful when ALL S1
    # compete; on a sample every target would pick among far fewer S1 records
    # and flood training with extra negatives -> off for sampled training
    sampled = bool(cfg.train_s1_sample) and len(s1) < n_s1_total
    tcfg = BlockConfig(n_jobs=n_jobs, reverse_k=0 if sampled else bcfg.reverse_k)
    A = featurize(s1, tgt, np.arange(len(s1)), tcfg, n_jobs, idf, cfg.batch_s1)
    del tgt
    tgt = A.pop("tgt")
    s1_ids = s1["entity_id"].to_numpy()
    t_ids = tgt["entity_id"].to_numpy()
    s1_ctry = s1["country"].astype(str).to_numpy()
    txt_s1, txt_t = (TextIndex(s1), TextIndex(tgt)) if cfg.use_nn else (None, None)
    del s1, tgt, idf
    clear_strcol_cache()
    free()

    truth = truth[truth["source1_entity_id"].isin(pd.Index(s1_ids))]
    s_index = pd.Series(np.arange(len(s1_ids)), index=s1_ids)
    n_true = np.bincount(s_index[truth["source1_entity_id"]].to_numpy(), minlength=len(s1_ids)).astype(float)
    key = pd.MultiIndex.from_arrays([s1_ids[A["s"]], t_ids[A["t"]]])
    y = key.isin(pd.MultiIndex.from_frame(truth)).astype(np.int8)
    del key, truth
    X, s_row, t_row = A["X"], A["s"], A["t"]
    reach = y.sum() / max(n_true.sum(), 1)
    log(f"  blocking recall ceiling: {reach:.4%} ({int(y.sum()):,}/{int(n_true.sum()):,}), "
        f"{len(y) / len(s1_ids):.1f} pairs per S1")
    dd = dict(zip(DOUBT_COLS, A["doubt"].T))
    weak_pos = ((y == 1) & (dd["n_tset_skel"] < 35) & (dd["n_idf_contain"] < 0.2) & (dd["a_tset"] < 40)
                & (dd["num_shared"] == 0) & (dd["n_alt_best"] < 60))
    twin_neg = ((y == 0) & (dd["n_exact_skel"] == 1) & (dd["a_missing_t"] == 0)
                & ((dd["a_tset"] >= 97) | ((dd["num_jacc"] >= 0.999) & (dd["num_s"] > 0))))
    ok = ~(weak_pos | twin_neg)
    log(f"  noise separation: {int(weak_pos.sum()):,} unlearnable positives and "
        f"{int(twin_neg.sum()):,} twin negatives removed from TRAINING only")
    del A, dd
    free()

    folds = list(GroupKFold(n_splits=cfg.folds).split(s_row, y, s_row))
    log("stage 1: LightGBM on all blocked pairs (OOF)")
    p_lgb1 = dict(objective="binary", learning_rate=0.1, num_leaves=127, min_data_in_leaf=50,
                  feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=1, lambda_l2=1.0,
                  verbose=-1, seed=cfg.seed, num_threads=n_jobs, force_col_wise=True)
    oof1, bst1 = lgb_oof(X, y, folds, ok, p_lgb1, 3000, "lgb-stage1", n_jobs)
    m1 = macro_micro(s_row, y, threshold_select(s_row, t_row, oof1, 0.5), n_true, cfg.beta)
    log(f"  stage-1 OOF @0.5: macro F0.5 {m1['macro_f05']:.5f}")

    # learned pruning: smallest candidate set keeping >= prune_keep of reachable links
    r1 = rank_in_s(s_row, oof1)
    best = None
    for K in (3, 4, 5, 6, 8, 10, 12, 16):
        for tau in (0.001, 0.003, 0.01, 0.02, 0.05):
            keep = (oof1 >= tau) & (r1 < K)
            if y[keep].sum() >= cfg.prune_keep * y.sum():
                if best is None or keep.sum() < best[2]:
                    best = (K, tau, int(keep.sum()))
    if best is None:
        best = (16, 0.001, int(((oof1 >= 0.001) & (r1 < 16)).sum()))
    K, tau, _ = best
    P = (oof1 >= tau) & (r1 < K)
    log(f"  pruning rule: rank < {K} and p1 >= {tau} -> {P.sum() / len(s1_ids):.2f} candidates per S1, "
        f"keeps {y[P].sum() / max(y.sum(), 1):.4%} of reachable links "
        f"(recall ceiling {y[P].sum() / max(n_true.sum(), 1):.4%})")
    report["train"] = {"blocking_recall": float(reach), "pairs_per_s1_blocking": float(len(y) / len(s1_ids)),
                       "prune_K": K, "prune_tau": tau, "cands_per_s1": float(P.sum() / len(s1_ids)),
                       "recall_after_prune": float(y[P].sum() / max(n_true.sum(), 1)),
                       "stage1_oof": m1}
    Xp, yp, sp, tp_, okp, p1p = X[P], y[P], s_row[P], t_row[P], ok[P], oof1[P]
    del X, y, s_row, t_row, ok, oof1, r1, folds
    free()
    bst1.save_model(str(out_dir / "lgb_stage1.txt"))
    save = dict(Xp=Xp, yp=yp, sp=sp, tp_=tp_, okp=okp, p1p=p1p, n_true=n_true,
                s1_ctry=np.asarray(s1_ctry, dtype=str))
    if txt_s1 is not None:
        save.update(txt_s1.to_arrays("s1_"))
        save.update(txt_t.to_arrays("t_"))
    np.savez(work / "A.npz", **save)
    (work / "A.json").write_text(json.dumps({"K": int(K), "tau": float(tau), "train": report["train"]},
                                            default=float))
    log(f"PHASE A done in {(time.time() - T0) / 60:.1f} min")


def phase_b(cfg: FinalConfig):
    import lightgbm as lgb
    T0 = time.time()
    n_jobs, out_dir, work = _setup(cfg)
    data_dir = auto_data_dir(cfg.data_dir, "test")
    bcfg = BlockConfig(n_jobs=n_jobs)
    meta = json.loads((work / "A.json").read_text())
    K, tau = meta["K"], meta["tau"]
    bst1 = lgb.Booster(model_file=str(out_dir / "lgb_stage1.txt"))
    report = {}
    # ===================== PHASE B: test (CPU) =====================
    log("PHASE B - test data")
    ts1, ttgt, _ = load_split(data_dir, "test")
    log(f"  S1={len(ts1):,} targets={len(ttgt):,}  countries={sorted(ts1['country'].astype(str).unique())}")
    normalize_frame(ts1, n_jobs)
    normalize_frame(ttgt, n_jobs)
    tidf = build_idf(ttgt)

    def scorer(Xb, sb):
        p = bst1.predict(Xb, num_threads=n_jobs).astype(np.float32)
        return (p >= tau) & (rank_in_s(sb, p) < K), p

    B = featurize(ts1, ttgt, np.arange(len(ts1)), bcfg, n_jobs, tidf, cfg.batch_s1, scorer=scorer)
    del ttgt
    ttgt = B.pop("tgt")
    ts1_ids = ts1["entity_id"].to_numpy()
    tt_ids = ttgt["entity_id"].to_numpy()
    ts1_ctry = ts1["country"].astype(str).to_numpy()
    txt_ts1, txt_tt = (TextIndex(ts1), TextIndex(ttgt)) if cfg.use_nn else (None, None)
    del ts1, ttgt, tidf
    clear_strcol_cache()
    free()
    Xt, st, tt, p1t = B["X"], B["s"], B["t"], B["p1"]
    log(f"  test: {B['n_block']:,} blocked pairs -> {len(st):,} candidates "
        f"({len(st) / len(ts1_ids):.2f} per S1)")
    cand_path = out_dir / "output" / "candidate_pairs.tsv"
    write_lists(cand_path, ts1_ids, st, tt_ids[tt], "candidate_entity_ids")
    report["test"] = {"n_s1": int(len(ts1_ids)), "blocked_pairs": int(B["n_block"]),
                      "cands_per_s1": float(len(st) / len(ts1_ids))}
    del B
    free()
    save = dict(Xt=Xt, st=st, tt=tt, p1t=p1t, ts1_ids=np.asarray(ts1_ids, dtype=str),
                tt_ids=np.asarray(tt_ids, dtype=str), ts1_ctry=np.asarray(ts1_ctry, dtype=str))
    if txt_ts1 is not None:
        save.update(txt_ts1.to_arrays("s1_"))
        save.update(txt_tt.to_arrays("t_"))
    np.savez(work / "B.npz", **save)
    (work / "B.json").write_text(json.dumps({"test": report["test"]}, default=float))
    log(f"PHASE B done in {(time.time() - T0) / 60:.1f} min")


def phase_c(cfg: FinalConfig):
    from sklearn.model_selection import GroupKFold
    T0 = time.time()
    n_jobs, out_dir, work = _setup(cfg)
    A = np.load(work / "A.npz")
    B = np.load(work / "B.npz")
    Xp, yp, sp, tp_, okp, p1p, n_true = (A[k] for k in ("Xp", "yp", "sp", "tp_", "okp", "p1p", "n_true"))
    s1_ctry = A["s1_ctry"].astype(object)
    Xt, st, tt, p1t = (B[k] for k in ("Xt", "st", "tt", "p1t"))
    ts1_ids, tt_ids = B["ts1_ids"].astype(object), B["tt_ids"].astype(object)
    ts1_ctry = B["ts1_ctry"].astype(object)
    if cfg.use_nn and "s1_nm" in A.files and "s1_nm" in B.files:
        txt_s1, txt_t = TextIndex.from_arrays(A, "s1_"), TextIndex.from_arrays(A, "t_")
        txt_ts1, txt_tt = TextIndex.from_arrays(B, "s1_"), TextIndex.from_arrays(B, "t_")
    else:
        txt_s1 = txt_t = txt_ts1 = txt_tt = None
    report = {"config": asdict(cfg)}
    report.update(json.loads((work / "A.json").read_text()))
    report.update(json.loads((work / "B.json").read_text()))
    cand_path = out_dir / "output" / "candidate_pairs.tsv"
    # ===================== PHASE C: ensemble (GPU allowed) =====================
    use_gpu = cfg.gpu == "on" or (cfg.gpu == "auto" and gpu_available())
    log(f"PHASE C - ensemble on pruned pairs (train {len(yp):,}, test {len(st):,}); GPU={use_gpu}")
    folds = list(GroupKFold(n_splits=cfg.folds).split(sp, yp, sp))
    members_oof = {"lgb1": p1p}
    members_test = {"lgb1": p1t}
    p_lgb2 = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_data_in_leaf=30,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=2.0,
                  verbose=-1, seed=cfg.seed + 1, num_threads=n_jobs, force_col_wise=True)
    o, b2 = lgb_oof(Xp, yp, folds, okp, p_lgb2, 4000, "lgb-stage2", n_jobs)
    members_oof["lgb2"], members_test["lgb2"] = o, b2.predict(Xt, num_threads=n_jobs).astype(np.float32)
    b2.save_model(str(out_dir / "lgb_stage2.txt"))
    for name, fn, flag in (("xgb", xgb_oof, cfg.use_xgboost), ("cat", cat_oof, cfg.use_catboost)):
        if not flag:
            continue
        try:
            members_oof[name], members_test[name] = fn(Xp, yp, folds, okp, Xt, use_gpu, n_jobs, cfg.seed)
        except Exception as e:  # GPU OOM / missing lib -> retry on CPU, else skip
            log(f"    {name} failed on {'GPU' if use_gpu else 'CPU'} ({type(e).__name__}: {e})")
            if use_gpu:
                try:
                    members_oof[name], members_test[name] = fn(Xp, yp, folds, okp, Xt, False, n_jobs, cfg.seed)
                except Exception as e2:
                    log(f"    {name} skipped ({type(e2).__name__}: {e2})")
    if cfg.use_nn and (use_gpu or cfg.nn_on_cpu):
        try:
            members_oof["charnn"], members_test["charnn"] = nn_oof(
                txt_s1, sp, txt_t, tp_, Xp, yp, folds, okp, txt_ts1, st, txt_tt, tt, Xt,
                gpu=use_gpu, n_jobs=n_jobs, seed=cfg.seed, budget_s=cfg.nn_budget_s)
        except Exception as e:  # never let the deep member break the run
            log(f"    charnn skipped ({type(e).__name__}: {e})")
    elif cfg.use_nn:
        log("    charnn skipped (no GPU)")
    del txt_s1, txt_t, txt_ts1, txt_tt
    free()
    names = list(members_oof)
    for nm in names:
        mm = macro_micro(sp, yp, threshold_select(sp, tp_, members_oof[nm], 0.5), n_true, cfg.beta)
        log(f"  member {nm:5s} OOF @0.5 macro F0.5 = {mm['macro_f05']:.5f}")

    def logit(p):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))

    blends = {nm: [nm] for nm in names}
    if len(names) > 1:
        blends["mean_all"] = names
        second = [n for n in names if n != "lgb1"]
        if len(second) > 1:
            blends["mean_stage2"] = second
        solo = {nm: macro_micro(sp, yp, threshold_select(sp, tp_, members_oof[nm], 0.5), n_true,
                                cfg.beta)["macro_f05"] for nm in names}
        top3 = sorted(names, key=lambda n: -solo[n])[:3]
        if len(top3) == 3:
            blends["mean_top3"] = top3
        if "charnn" in names:
            blends["gbdt_best+charnn"] = [max((n for n in names if n != "charnn"), key=lambda n: solo[n]), "charnn"]
    missed = max(float(n_true.sum() - yp.sum()), 0.0)
    lam_hat = missed / len(n_true)
    results = {}
    for bname, mem in blends.items():
        z = np.mean([logit(members_oof[m]) for m in mem], axis=0)
        pb = 1 / (1 + np.exp(-z))
        cal = Calibrator().fit(pb, yp)
        q = cal(pb)
        rule, m = tune_rule(sp, tp_, yp, q, n_true, cfg.beta, lam_hat)
        results[bname] = (m["macro_f05"], rule, cal, mem, m)
        log(f"  blend {bname:12s} OOF macro F0.5 = {m['macro_f05']:.5f}  rule={rule}")
    bname = max(results, key=lambda k: results[k][0])
    score, rule, cal, mem, m = results[bname]
    log(f"CHOSEN: blend={bname} {mem} rule={rule}")
    log(f"  OOF (all sampled train S1, blocking + pruning losses counted): "
        f"macro F0.5={m['macro_f05']:.5f}  macro P={m['macro_p']:.5f}  macro R={m['macro_r']:.5f}  "
        f"micro F0.5={m['micro_f05']:.5f}")
    report["oof"] = {"chosen_blend": bname, "members": mem, "rule": rule, **m,
                     "all_blends": {k: v[0] for k, v in results.items()}}

    # per-country decision rules (countries seen in training); unseen
    # countries such as France keep the global rule
    zq = np.mean([logit(members_oof[mm]) for mm in mem], axis=0)
    q_oof = cal(1 / (1 + np.exp(-zq)))
    country_rules = {}
    if cfg.per_country_rules:
        row_ctry = s1_ctry[sp]
        for c in sorted(set(s1_ctry)):
            univ = s1_ctry == c
            if univ.sum() < 2000:
                continue
            rm = row_ctry == c
            g = macro_micro(sp[rm], yp[rm], apply_rule(rule, sp[rm], tp_[rm], q_oof[rm], cfg.beta),
                            n_true, cfg.beta, univ)["macro_f05"]
            rc, mc = tune_rule(sp[rm], tp_[rm], yp[rm], q_oof[rm], n_true, cfg.beta, lam_hat, univ)
            log(f"  country {c:8s}: global rule {g:.5f} -> own rule {mc['macro_f05']:.5f} {rc}")
            if mc["macro_f05"] > g + 2e-4:
                country_rules[c] = rc
        if country_rules:
            sel_oof = np.zeros(len(yp), bool)
            row_ctry = s1_ctry[sp]
            for c in set(row_ctry):
                rm = row_ctry == c
                sel_oof[rm] = apply_rule(country_rules.get(c, rule), sp[rm], tp_[rm], q_oof[rm], cfg.beta)
            m2 = macro_micro(sp, yp, sel_oof, n_true, cfg.beta)
            log(f"  per-country rules: OOF macro F0.5 {m['macro_f05']:.5f} -> {m2['macro_f05']:.5f}")
            if m2["macro_f05"] > m["macro_f05"]:
                m = m2
                report["oof"].update({**m2, "country_rules": country_rules})
            else:
                country_rules = {}

    zt = np.mean([logit(members_test[mm]) for mm in mem], axis=0)
    qt = cal(1 / (1 + np.exp(-zt)))
    sel = np.zeros(len(st), bool)
    t_ctry_rows = ts1_ctry[st]
    for c in set(t_ctry_rows):
        rm = t_ctry_rows == c
        sel[rm] = apply_rule(country_rules.get(c, rule), st[rm], tt[rm], qt[rm], cfg.beta)
    match_path = out_dir / "output" / "matching_results.tsv"
    res = write_lists(match_path, ts1_ids, st[sel], tt_ids[tt[sel]], "matched_entity_ids")
    n_links = int(sel.sum())
    empty = float((res["matched_entity_ids"] == "").mean())
    log(f"  test: {n_links:,} links, {empty:.2%} S1 predicted as singletons")
    report["test"].update({"links": n_links, "pred_singleton_rate": empty})

    issues = validate_outputs(match_path, cand_path, ts1_ids, tt_ids)
    report["validation"] = issues or "PASS"
    log("VALIDATION: " + ("PASS" if not issues else "; ".join(issues)))
    if is_kaggle():
        shutil.copy(match_path, "/kaggle/working/submission.tsv")
    report["runtime_min"] = round((time.time() - T0) / 60, 1)
    (out_dir / "report.json").write_text(json.dumps(report, indent=2, default=str))
    log(f"done in {report['runtime_min']} min -> {match_path}")
    return report


def run_final(cfg: FinalConfig):
    """Runs phases A, B, C in three fresh processes: each starts with clean
    memory (no fragmentation carried over from the previous phase) and CUDA
    is only ever initialised in phase C, after which nothing forks."""
    import subprocess
    import sys
    T0 = time.time()
    _, out_dir, work = _setup(cfg)
    cfg_path = work / "config.json"
    cfg_path.write_text(json.dumps(asdict(cfg)))
    env = dict(os.environ, PYTHONPATH=str(Path(__file__).resolve().parent.parent)
               + os.pathsep + os.environ.get("PYTHONPATH", ""))
    for ph in ("A", "B", "C"):
        log(f"===== starting phase {ph} in a fresh process =====")
        r = subprocess.run([sys.executable, "-u", "-m", "entity_matching.final", "--phase", ph,
                            "--cfg", str(cfg_path)], env=env)
        if r.returncode != 0:
            raise RuntimeError(f"phase {ph} failed with exit code {r.returncode}"
                               + (" (killed: out of memory)" if r.returncode == -9 else ""))
    log(f"all phases done in {(time.time() - T0) / 60:.1f} min")
    shutil.rmtree(work, ignore_errors=True)  # large hand-over files; outputs are kept
    return json.loads((out_dir / "report.json").read_text())


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--phase", choices=["A", "B", "C"], required=True)
    ap.add_argument("--cfg", required=True)
    a = ap.parse_args()
    c = FinalConfig(**json.loads(Path(a.cfg).read_text()))
    {"A": phase_a, "B": phase_b, "C": phase_c}[a.phase](c)

In [ ]:
%%writefile run_final.py
"""CLI for the final competition pipeline (entity_matching/final.py).

    python run_final.py --team MYTEAM                 # auto-detects /kaggle/input data
    python run_final.py --data-dir dataset --out-dir out --gpu off
"""
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from entity_matching.final import FinalConfig, run_final  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--data-dir", default=None)
ap.add_argument("--out-dir", default=None)
ap.add_argument("--train-s1-sample", type=int, default=100_000)
ap.add_argument("--folds", type=int, default=3)
ap.add_argument("--gpu", choices=["auto", "on", "off"], default="auto")
ap.add_argument("--team", default="team")
ap.add_argument("--no-xgb", action="store_true")
ap.add_argument("--no-cat", action="store_true")
ap.add_argument("--n-jobs", type=int, default=None)
a = ap.parse_args()
run_final(FinalConfig(data_dir=a.data_dir, out_dir=a.out_dir, train_s1_sample=a.train_s1_sample,
                      folds=a.folds, gpu=a.gpu, team_name=a.team, use_xgboost=not a.no_xgb,
                      use_catboost=not a.no_cat, n_jobs=a.n_jobs))

## 3 · Sanity check: hardcoded noise patterns on real true pairs from the EDA

In [ ]:
import pandas as pd
from entity_matching.normalize import normalize_name
pairs = [("Gold Solutions", "গোল্ড সলিউশনস"), ("Jain Builders Private Limited", "जैन बिल्डर्स प्राइवेट लिमिटेड"),
         ("Creative Builders", "ಕ್ರಿಯೇಟಿವ್ ಬಿಲ್ಡರ್ಸ್"), ("Arihant Technology Limited", "Arihant टेक्नोलॉजी लिमिटेड"),
         ("Creative Intermediate LLC", "Llc Creative Intermediate"), ("South Academy", "5outh Academy"),
         ("Washington, Harris and Burrell", "WASHINGT0N, HARRIS AND BURRELL"),
         ("Midwest Applied, LLC", "Midwest Applied, ([LLC])"),
         ("Fort Worth Telecommunication", "Fort Worth Telecommunication L.L.C. (ID: 49166)"),
         ("Summit Ministries", "Summit Ministries lnc"), ("Bright Automation Enterprises", "Bright Bright Automation Enterprises")]
df = pd.DataFrame([{"S1": a, "S2/S3": b, "key S1": normalize_name(a)["skel"], "key S2/S3": normalize_name(b)["skel"]}
                   for a, b in pairs])
df["match"] = df["key S1"] == df["key S2/S3"]
print(f"{df['match'].sum()}/{len(df)} real true pairs normalise to the same key"); df

## 4 · Smoke test (synthetic data, separate process)
Runs the complete pipeline on 2k synthetic S1 records with the same folder layout, including France in test. It exercises
blocking, pruning, the GPU/CPU ensemble, validation and output writing in about a minute, so an environment problem
shows up **before** the long run. It runs in a subprocess, so the notebook kernel never touches CUDA.

In [ ]:
import shutil
if RUN_SMOKE_TEST:
    shutil.rmtree("_smoke", ignore_errors=True)
    r = subprocess.run([sys.executable, "-c",
        "from entity_matching.synth import write_synthetic; write_synthetic('_smoke/data', 2000, 1000, 7)"],
        capture_output=True, text=True); print(r.stdout[-500:], r.stderr[-2000:])
    r = subprocess.run([sys.executable, "run_final.py", "--data-dir", "_smoke/data", "--out-dir", "_smoke/out",
                        "--train-s1-sample", "0", "--gpu", GPU], capture_output=True, text=True)
    tail = "\n".join(l for l in r.stdout.splitlines() if "features batch" not in l)
    print(tail[-3500:]); print(r.stderr[-3000:])
    assert r.returncode == 0 and "VALIDATION: PASS" in r.stdout, "SMOKE TEST FAILED - fix before the real run"
    shutil.rmtree("_smoke", ignore_errors=True)
    for f in ("submission.tsv",):  # never leave a synthetic file behind
        if os.path.exists(f): os.remove(f)
    print("SMOKE TEST PASSED")

## 5 · Real run (fresh subprocess, live log)
The pipeline runs as **three fresh processes**: A (train), B (test), then C (ensemble, GPU). Each starts with clean memory,
and CUDA is initialised only in C. Rough runtime on Kaggle: A ~25 min, B ~35 min, C ~10 min.
The log shows RAM per step, the blocking recall ceiling, candidates per S1, the OOF score of every member and blend,
and the final validation.

In [ ]:
cmd = [sys.executable, "-u", "run_final.py", "--train-s1-sample", str(TRAIN_S1_SAMPLE),
       "--folds", str(FOLDS), "--gpu", GPU, "--team", TEAM_NAME, "--out-dir", WORK]
if DATA_DIR: cmd += ["--data-dir", DATA_DIR]
with open("run_log.txt", "w") as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        logf.write(line)
        print(line, end="")  # every line: progress is always visible
    proc.wait()
print("exit code:", proc.returncode)
assert proc.returncode == 0, "run failed - see run_log.txt"

## 6 · Results

In [ ]:
import json
rep = json.load(open(os.path.join(WORK, "report.json")))
print("VALIDATION:", rep["validation"])
print(f"train: blocking recall {rep['train']['blocking_recall']:.4%} | after pruning {rep['train']['recall_after_prune']:.4%} "
      f"| candidates/S1 {rep['train']['cands_per_s1']:.2f}")
print(f"test : {rep['test']['n_s1']:,} S1 | candidates/S1 {rep['test']['cands_per_s1']:.2f} | links {rep['test']['links']:,} "
      f"| predicted singletons {rep['test']['pred_singleton_rate']:.2%}")
o = rep["oof"]
print(f"OOF  : blend={o['chosen_blend']} rule={o['rule']}")
print(f"       macro F0.5 {o['macro_f05']:.5f} | macro P {o['macro_p']:.5f} | macro R {o['macro_r']:.5f}")
display(pd.Series(o["all_blends"], name="OOF macro F0.5").sort_values(ascending=False).to_frame())
for f in ("matching_results.tsv", "candidate_pairs.tsv"):
    display(pd.read_csv(os.path.join(WORK, "output", f), sep="\t", dtype=str, keep_default_na=False).head(5))

## 7 · Final submission package (zip)

In [ ]:
import importlib.metadata as im, zipfile, datetime
pkg_root = os.path.join(WORK, "code", "business_entity_resolution")
shutil.rmtree(os.path.join(WORK, "code"), ignore_errors=True)
os.makedirs(os.path.join(pkg_root, "src"), exist_ok=True)
shutil.copytree("entity_matching", os.path.join(pkg_root, "src", "entity_matching"),
                ignore=shutil.ignore_patterns("__pycache__"))
shutil.copy("run_final.py", os.path.join(pkg_root, "src", "run_final.py"))
vers = []
for p in ["pandas", "numpy", "scikit-learn", "lightgbm", "xgboost", "catboost", "rapidfuzz", "anyascii", "torch"]:
    try: vers.append(f"{p}=={im.version(p)}")
    except Exception: pass
open(os.path.join(pkg_root, "requirements.txt"), "w").write("\n".join(vers) + "\n")
open(os.path.join(pkg_root, "README.md"), "w").write(f'''# Business Entity Resolution — reproduction

```bash
pip install -r requirements.txt
cd src
python run_final.py --data-dir <path to dataset/> --out-dir <out> --train-s1-sample {TRAIN_S1_SAMPLE} --folds {FOLDS} --gpu auto
```
Writes `<out>/output/matching_results.tsv`, `<out>/output/candidate_pairs.tsv`, `<out>/report.json`.
`--gpu off` runs everything on CPU. The pipeline is deterministic given the seed (42).
Pipeline: normalise -> block -> features -> stage-1 LightGBM -> learned pruning (candidate_pairs.tsv)
-> LightGBM + XGBoost + CatBoost ensemble -> calibration -> macro-F0.5 decision (matching_results.tsv).
''')
t, o = rep["train"], rep["oof"]
doc = f'''# Methodology — Amazon ML Challenge 2026: Business Entity Resolution
Team: {TEAM_NAME} · generated {datetime.date.today()}

## 1. Methodology
Source 1 is the deduplicated reference. For each S1 entity we (1) generate candidates from S2/S3, (2) score them with
a learned pairwise model, (3) prune to a small candidate set, (4) re-score with a gradient-boosting ensemble and
(5) choose the final set per S1 to maximise the **macro F0.5** (the official metric; singletons count as 1.0 when
predicted empty).

Data facts used (EDA): country is a hard boundary (100% of true links are same-country); every S2/S3 record belongs
to at most one S1 (one-owner constraint, enforced at decision time); 5.6% of S1 entities are singletons; France appears
only in test, so no rule depends on the countries seen in training.

**Noise normalisation (hardcoded patterns):**
- Indic scripts: native legal words are mapped (प्राइवेट→private …), the rest is transliterated, then a
  phonetic consonant skeleton is taken so `গোল্ড সলিউশনস` = `Gold Solutions`.
- Legal forms (US/IN/FR, dotted, bracketed, moved) are canonicalised and separated from the core name.
- OCR digits (5outh, WASHINGT0N, lnc) are repaired; accents folded.
- Junk (`***`, `(ID: 123)`, `M/s`), domains (aahanarealty.com) and DBA/formerly aliases are handled.
- Addresses: `NULL` placeholders, state names/codes (including native script), street and unit abbreviations,
  city aliases, canonical numbers (leading zeros, ordinals, compound numbers).

**Noise separation:** unlearnable positives (no name, address or number evidence) and twin negatives
(indistinguishable from a link) are removed from training folds only.

## 2. Candidate generation / blocking
- Country-partitioned inverted index over hashed keys, weighted by IDF. Key types: name skeleton tokens, bigrams,
  8-character space-less prefix, full name key, address words, canonical numbers, number+street, and
  **name-token × number** composites (rare even for generic names like "Pediatric Group").
- Each key has a document-frequency cap. Per S1, keys are used rarest-first under a posting budget.
- Forward adaptive-K (keep 12, up to 60 while score ≥ 0.25 × best), plus top-10 by name-only and address-only
  score, plus **reverse search** (each target keeps its top-5 S1).
- Blocking recall on the training sample: **{t["blocking_recall"]:.4%}** at {t["pairs_per_s1_blocking"]:.1f} pairs per S1.
- **Learned pruning:** stage-1 LightGBM scores all blocked pairs. We keep rank < {t["prune_K"]} and
  p ≥ {t["prune_tau"]}, the smallest set retaining ≥ 99.8% of reachable true links, chosen on out-of-fold
  predictions. Result: **{t["cands_per_s1"]:.2f} candidates per S1 (train, OOF)** and
  **{rep["test"]["cands_per_s1"]:.2f} per S1 on test**, recall ceiling {t["recall_after_prune"]:.4%}.
  `candidate_pairs.tsv` is exactly this pruned set, which is what the final ensemble scores.

## 3. Model architecture and features
- 75 features: fuzzy ratios over five name views (core / OCR-folded / phonetic / space-less / full), IDF-weighted
  token overlap, alias, domain, acronym and legal-form agreement, name frequency; address fuzzy and IDF overlap,
  number agreement (exact / compound / parts / house number / number+street, **rarity-weighted shared and
  conflicting numbers**), state and postal agreement; hardcoded rule flags; per-S1 rank/gap/sibling context.
- Stage 1: LightGBM (GroupKFold by S1). Stage 2 on pruned pairs: LightGBM + XGBoost (CUDA) + CatBoost (GPU)
  + **charnn**, a deep character-level cross-attention matcher: 96-byte name+address sequences, a shared residual
  char-CNN, ESIM-style soft alignment with [h−a, h⊙a] comparison, mean/max pooling, fused with the 75
  standardised tabular features. It is trained fold-wise with mixed precision and a wall-clock budget. All members
  produce OOF predictions; the blend (single, logit-mean of all / top-3, best GBDT + charnn) is chosen on OOF
  macro F0.5, then isotonic calibration.
- Decision: best on OOF of a threshold rule or the per-S1 **expected-F0.5 set** (plug-in General F-measure Maximizer:
  choose k maximising 1.25·Σq/(0.25·E|T|+k) against P(empty)), with one owner per target. A country's own rule
  is used only when it beats the global rule on that country's OOF by > 0.0002; France (unseen) uses the global rule.
- Country rules used: {o.get("country_rules", "none (global rule)")}.
- Chosen: blend **{o["chosen_blend"]}** ({", ".join(o["members"])}), rule {o["rule"]}.

## 4. Validation
GroupKFold by S1 on {rep["config"]["train_s1_sample"] or "all"} training S1 records. Every held-out S1 is scored, with
true links lost to blocking or pruning counted as false negatives.
**OOF macro F0.5 = {o["macro_f05"]:.5f}** (macro P {o["macro_p"]:.5f}, macro R {o["macro_r"]:.5f}).
Blend comparison: {", ".join(f"{k} {v:.5f}" for k, v in o["all_blends"].items())}.

## 5. Other
No external data, APIs or lookups. Models are LightGBM (MIT), XGBoost (Apache-2.0), CatBoost (Apache-2.0),
and a small PyTorch network (BSD-style licence, trained from scratch, about 0.37M parameters, far below 8B).
The pipeline is deadlock-safe (all forks before CUDA; pool timeouts) and bounded in RAM (batching, posting budget).
'''
open(os.path.join(WORK, "Documentation_template.md"), "w").write(doc)
zpath = os.path.join(WORK, f"{TEAM_NAME}_submission.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in ["output/matching_results.tsv", "output/candidate_pairs.tsv", "Documentation_template.md"]:
        z.write(os.path.join(WORK, rel), rel)
    for root, _, files in os.walk(os.path.join(WORK, "code")):
        for f in files:
            full = os.path.join(root, f); z.write(full, os.path.relpath(full, WORK))
print("package:", zpath, f"{os.path.getsize(zpath) / 1e6:.1f} MB")
with zipfile.ZipFile(zpath) as z:
    print("\n".join(z.namelist()[:40]))